# 🤖 43 · Foundation models, language models and AI agents for biology
### What is inside a large language model, how retrieval and tool use work, how agents and "AI scientists" fail, and how to evaluate biological foundation models fairly — every piece rebuilt at toy scale with a known ground truth

**BiologyToML** · Level 2 · Research frontier · Part I — Causality, reliability and AI-driven discovery · Module 43 of 44

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/43_llms_ai_agents.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1–2 min to run everything); no GPU, no internet, no API key needed |
| **Core frameworks** | Python 3, NumPy, SciPy, pandas, scikit-learn, PyTorch (CPU), Matplotlib |
| **Prerequisites** | Level 1 (especially Modules 19–20, 22 and 23) and [Module 26](26_attention_transformers.ipynb) (attention and transformers); helpful: Modules 24/30 (pseudobulk differential expression) and 42 (trustworthy evaluation) |
| **Estimated time** | 190 min (reading + running + the ✏️ exercises) |
| **Builds on** | [19 · HMM inference](19_hmm_inference.ipynb), [20 · HMM learning](20_hmm_learning.ipynb), [22 · Neural networks](22_neural_networks.ipynb), [23 · Model selection](23_model_selection.ipynb), [26 · Attention and transformers](26_attention_transformers.ipynb), [42 · Trustworthy ML](42_trustworthy_ml.ipynb) |
| **Research frontier covered** | large language models (pre-training, scaling, instruction tuning), retrieval-augmented generation, tool-using AI agents and AI scientists, biological foundation models and their evaluation, responsible use — see [`FRONTIER_MAP.md`](../FRONTIER_MAP.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> Everything is **SIMULATED and offline**. The "language" is a corpus of lab-notebook sentences generated by a small
> probabilistic grammar from a hidden table of facts (which stimulus induces which gene in which cell type), so the true
> probability of every word is known. The "literature" is 200 synthetic abstracts with known topics and known facts. The
> agent analyses `simulate_scrna` data (true responding genes known) and is driven by a **scripted, rule-based policy**
> standing in for an LLM — no external model or API is called; how to plug in a real one is shown as non-executed
> pseudo-code. The protein foundation model is a small masked language model trained on a simulated protein family.
> Real LLMs are about a million times larger and are trained on human-written text — Section 9 explains what changes.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** language modelling as next-token prediction, **build** n-gram models as Markov chains, and **compute** cross-entropy and perplexity.
2. **Implement** a small decoder-only transformer (causal self-attention) in PyTorch and **compare** it, word slot by word slot, with n-grams and the true generating grammar.
3. **Explain** temperature sampling, the qualitative scaling of loss with model size, and **demonstrate** hallucination as fluent but ungrounded generation.
4. **Implement** byte-pair encoding and **describe** the stages from pre-training to instruction tuning and preference optimisation.
5. **Build** TF-IDF and latent-semantic retrieval, **evaluate** it with precision@k, and **build** a retrieval-augmented answer composer whose claims are verified against the retrieved text.
6. **Implement** a ReAct-style agent loop with tools (literature search, differential expression, marker lookup), **evaluate** its answer against ground truth and **add guardrails** against its failure modes.
7. **Evaluate** a biological foundation model fairly: frozen embeddings and zero-shot scores of a masked protein language model against simple baselines under proper splits.
8. **Discuss** AI-scientist systems, automated false discovery, biosecurity and privacy, and **apply** a checklist for using LLMs responsibly in research.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, PyTorch, course toolkit | 5 min |
| 1 · Language modelling = next-token prediction | a simulated lab-notebook language; n-grams; cross-entropy, perplexity | 20 min |
| 2 · 🧪 A tiny GPT | causal attention; slot-by-slot ladder vs oracle; scaling; temperature; hallucination | 30 min |
| 3 · Tokenisation and training stages | BPE on DNA learns motifs; pre-training → instruction tuning → preferences | 15 min |
| 4 · 🧪 Embeddings and retrieval | 200 synthetic abstracts; TF-IDF, LSA; precision@k | 20 min |
| 5 · Retrieval-augmented generation | cited answers; a retrieval miss → confident wrong answer; verification | 20 min |
| 6 · 🧪 Agents | ReAct loop with tools; scripted policy; ground-truth check; failure modes and guardrails | 30 min |
| 7 · 🧪 Biological foundation models | a masked protein LM vs one-hot/PCA baselines; zero-shot scores | 20 min |
| 8 · AI scientists and lab automation | automated false discovery; evaluation; biosecurity and privacy | 15 min |
| 9 · 🧭 From toy to real systems | real models and resources; using LLMs responsibly in research | 10 min |
| ✅ / 🧠 / 🚀 / 📝 | takeaways, checkpoint, discussion, going further, lab note | 5 min |

## 🧫 The biological question

Biologists now use large language models (LLMs) every day: to summarise papers, write analysis code, draft protocols and
brainstorm hypotheses. Systems built around LLMs go further. *Coscientist* planned and executed chemistry experiments on
robotic hardware (Boiko *et al.* 2023); *ChemCrow* gave an LLM access to chemistry tools (Bran *et al.* 2024); the
*Virtual Lab* had LLM "agents" with different roles design nanobodies against SARS-CoV-2 variants, some of which were then
validated experimentally (Swanson *et al.* 2025); and an "AI co-scientist" generates and ranks research hypotheses
(Gottweis *et al.* 2025, preprint). In parallel, **biological foundation models** — transformers pre-trained on millions
of protein sequences, genomes or single-cell transcriptomes — are offered as general-purpose starting points for many
tasks ([Module 26](26_attention_transformers.ipynb)).

These systems are powerful and fallible in specific, predictable ways: they produce fluent text that is not grounded in
evidence, cite papers that do not exist, call the wrong tool, repeat themselves, and are sometimes beaten by simple
baselines. To use them well a biologist needs to know **what is inside**.

This notebook opens each box at toy scale, with a known truth: a tiny GPT trained on SIMULATED lab-notebook sentences whose
true probabilities we know; a retrieval engine and a retrieval-augmented answerer over 200 synthetic abstracts; a ReAct
agent that runs a differential-expression analysis on `simulate_scrna` data and checks the literature; a masked protein
language model compared fairly with simple baselines; and an "automated scientist" that shows how automation scales false
discoveries as easily as true ones.

**What is inside these systems, how do they fail, and how should a biologist use and evaluate them?**

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this notebook uses PyTorch (preinstalled on Colab); a CPU is enough.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sklearn", "scikit-learn"), ("torch", "torch")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import math
import re
import time
from collections import Counter, defaultdict
from itertools import product

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import stats
from sklearn.linear_model import RidgeCV

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

In [ ]:
# ── 0.1 PyTorch configuration ──
torch.manual_seed(SEED)                  # PyTorch has its own random-number generator
torch.set_num_threads(1)                 # tiny models: one thread avoids overhead (and is polite on shared CPUs)
OKABE_ITO = ["#0072B2", "#D55E00", "#009E73", "#E69F00", "#CC79A7", "#56B4E9", "#000000"]   # colour-blind-safe palette
T_START = time.time()
print("PyTorch", torch.__version__, "| threads:", torch.get_num_threads(), "| device: cpu")

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
We use the single-cell simulator (the agent's laboratory data), regulatory DNA (for tokenisation), the helix-bundle structure and Potts score (for the protein language model), Benjamini–Hochberg correction, PCA and the embedding plot. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MITO_GENES = ["MT-ND1", "MT-ND2", "MT-CO1", "MT-CO2", "MT-ATP8", "MT-ATP6", "MT-CO3", "MT-ND3",
              "MT-ND4L", "MT-ND4", "MT-ND5", "MT-ND6", "MT-CYB"]


INTERFERON_GENES = ["ISG15", "IFI6", "IFIT1", "IFIT3", "MX1", "OAS1", "IFI44L", "RSAD2", "IRF7", "STAT1"]


CYCLING_GENES = ["MKI67", "TOP2A", "CCNB1", "CDK1", "UBE2C", "BIRC5", "TYMS", "PCNA"]


PBMC_TYPES = {
    "CD4 T": ("lymphoid", ["CD3E", "CD3D", "IL7R", "CD4"], 0.5),
    "CD8 T": ("lymphoid", ["CD3E", "CD3D", "CD8A", "CD8B"], 0.5),
    "NK": ("lymphoid", ["NKG7", "GNLY", "KLRD1", "PRF1"], 2.2),
    "B": ("lymphoid", ["MS4A1", "CD79A", "CD79B", "CD19"], 2.4),
    "Monocyte": ("myeloid", ["CD14", "LYZ", "S100A8", "S100A9"], 2.6),
    "Dendritic": ("myeloid", ["FCER1A", "CST3", "CLEC10A", "HLA-DQA1"], 2.2),
}


def sample_negative_binomial(rng: np.random.Generator, mu, theta):
    """Negative-binomial counts with mean ``mu`` and inverse dispersion ``theta``.

    Variance = mu + mu²/theta (large theta → Poisson).  Drawn as a Gamma–Poisson mixture:
    each count is Poisson with a rate that itself varies from cell to cell.
    """
    mu = np.asarray(mu, float)
    theta = np.broadcast_to(np.asarray(theta, float), mu.shape)
    return rng.poisson(rng.gamma(theta, np.where(mu > 0, mu, 0.0) / theta)).astype(np.int32)


def make_expression_programs(rng: np.random.Generator, n_genes: int, cell_types: dict,
                             n_extra_markers: int = 20, extra_names=()):
    """Log-scale expression profiles (one row per cell type) for a simulated tissue.

    ``cell_types`` maps a type name to (lineage, named marker genes, marker strength).  Every
    type gets its named markers plus ``n_extra_markers`` anonymous ones; types of the same
    lineage share 20 lineage genes.  ``extra_names`` are reserved gene names (placed after the
    markers) that the caller gives its own behaviour.  Returns ``(log_profiles, genes, marker_of)``.
    """
    named = []
    for _, markers, _ in cell_types.values():
        named += [g for g in markers if g not in named]
    reserved = named + [g for g in extra_names if g not in named]
    if len(reserved) > n_genes:
        raise ValueError("n_genes is too small for the requested marker genes")
    genes = np.array(reserved + [f"Gene{j:04d}" for j in range(1, n_genes - len(reserved) + 1)], dtype=object)
    free = np.arange(len(reserved), n_genes)                       # anonymous genes available
    base = rng.normal(0.0, 2.0, n_genes)                           # gene-specific baseline (log scale)
    base[:len(named)] -= 2.0                                       # named markers are low elsewhere
    profiles = np.tile(base, (len(cell_types), 1)) + rng.normal(0, 0.15, (len(cell_types), n_genes))
    marker_of = np.array([""] * n_genes, dtype=object)
    lineages = sorted({v[0] for v in cell_types.values()})
    lineage_genes = {lin: rng.choice(free, 20, replace=False) for lin in lineages}
    for t, (name, (lineage, markers, strength)) in enumerate(cell_types.items()):
        profiles[t, lineage_genes[lineage]] += rng.normal(1.2, 0.3, 20)
        idx = [int(np.flatnonzero(genes == g)[0]) for g in markers]
        profiles[t, idx] += strength + 2.0 + rng.normal(0, 0.2, len(idx))
        extra = rng.choice(free, n_extra_markers, replace=False)
        profiles[t, extra] += rng.normal(0.6 + strength / 2, 0.3, n_extra_markers)
        for j in list(idx) + list(extra):
            marker_of[j] = name if marker_of[j] == "" else marker_of[j] + "/" + name
    return profiles, genes, marker_of


def simulate_scrna(rng: np.random.Generator, n_cells: int = 3000, n_genes: int = 1000, cell_types=None,
                   proportions=None, n_donors: int = 6, n_batches: int = 2, confounded: bool = False,
                   depth: float = 1500, batch_effect: float = 0.25, donor_effect: float = 0.15,
                   stim_effect: float = 1.0, theta: float = 8.0, doublet_rate: float = 0.0,
                   damaged_rate: float = 0.0, cycling_rate: float = 0.0, drop_type_in_batch=None) -> dict:
    """Simulate single-cell RNA-seq UMI counts for a PBMC-like sample collection.

    Design: ``n_donors`` donors, the first half *control* and the second half *stimulated*
    (an interferon response in every cell type plus a monocyte-specific response).  Donor
    ``d`` is sequenced in batch ``d % n_batches`` — or, with ``confounded=True``, every control
    donor in batch 1 and every stimulated donor in batch 2 (a design flaw to learn from).
    Counts are negative binomial around ``depth`` UMIs per cell.  Optional artefacts for
    quality control: doublets, damaged cells (high mitochondrial fraction), cycling cells.
    ``drop_type_in_batch=("B", 1)`` removes B cells from batch 1 (index 1, i.e. "batch2").

    Expression probabilities are renormalised within each cell (a fixed-size library, as in
    sequencing), so in stimulated cells the genes with ``true_logfc == 0`` lose a few per cent of
    their share (about −0.04 on the natural-log scale) to the induced genes: a compositional
    effect that is also present in real data.

    Returns a dict with ``counts`` (cells × genes, int32), ``obs`` (per-cell table with the
    true cell type, donor, batch, condition, …), ``var`` (per-gene table), ``true_logfc``
    (cell type × gene stimulation effect, natural log) and ``log_profiles``.
    """
    cell_types = PBMC_TYPES if cell_types is None else cell_types
    names = list(cell_types)
    T = len(names)
    p = np.full(T, 1 / T) if proportions is None else np.asarray(proportions, float) / np.sum(proportions)
    extra = MITO_GENES + INTERFERON_GENES + CYCLING_GENES
    profiles, genes, marker_of = make_expression_programs(rng, n_genes, cell_types, extra_names=extra)
    G = len(genes)
    is_mito, is_ifn, is_cyc = (np.isin(genes, MITO_GENES), np.isin(genes, INTERFERON_GENES),
                               np.isin(genes, CYCLING_GENES))
    profiles[:, is_mito] = rng.normal(3.0, 0.3, is_mito.sum())          # ≈ 4 % of UMIs are mitochondrial
    profiles[:, is_ifn | is_cyc] = rng.normal(-1.0, 0.4, (T, (is_ifn | is_cyc).sum()))
    # stimulation: interferon genes up everywhere + 40 monocyte-specific responders (up or down)
    true_logfc = np.zeros((T, G))
    true_logfc[:, is_ifn] = stim_effect * rng.normal(1.6, 0.3, is_ifn.sum())
    if "Monocyte" in names:
        responders = rng.choice(np.flatnonzero((marker_of == "") & ~is_mito & ~is_ifn & ~is_cyc), 40, replace=False)
        true_logfc[names.index("Monocyte"), responders] = stim_effect * rng.choice([-1, 1], 40) * rng.normal(1.2, 0.3, 40)
    # donors, batches, conditions
    donor = rng.integers(0, n_donors, n_cells)
    stimulated = donor >= n_donors // 2
    batch = stimulated.astype(int) if confounded else donor % n_batches
    batch = np.minimum(batch, n_batches - 1)
    cell_type = rng.choice(T, n_cells, p=p)
    if drop_type_in_batch is not None:
        t_drop, b_drop = names.index(drop_type_in_batch[0]), drop_type_in_batch[1]
        bad = (cell_type == t_drop) & (batch == b_drop)
        cell_type[bad] = rng.choice(np.delete(np.arange(T), t_drop), bad.sum(), p=np.delete(p, t_drop) / (1 - p[t_drop]))
    batch_shift = rng.normal(0, batch_effect, (n_batches, G))
    batch_depth = np.exp(rng.normal(0, 0.3, n_batches))
    donor_shift = rng.normal(0, donor_effect, (n_donors, G))
    log_rate = (profiles[cell_type] + batch_shift[batch] + donor_shift[donor]
                + true_logfc[cell_type] * stimulated[:, None])
    cycling = rng.random(n_cells) < cycling_rate
    log_rate[np.ix_(cycling, np.flatnonzero(is_cyc))] += 3.0
    damaged = rng.random(n_cells) < damaged_rate
    log_rate[np.ix_(damaged, np.flatnonzero(is_mito))] += 2.2
    lib = depth * batch_depth[batch] * np.exp(rng.normal(0, 0.35, n_cells))
    lib[damaged] *= 0.35
    prob = np.exp(log_rate - log_rate.max(axis=1, keepdims=True))
    prob /= prob.sum(axis=1, keepdims=True)
    theta_g = theta * np.exp(rng.normal(0, 0.5, G))
    mu = lib[:, None] * prob
    doublet = rng.random(n_cells) < doublet_rate
    partner = rng.integers(0, n_cells, n_cells)
    mu[doublet] = 0.9 * mu[doublet] + 0.9 * mu[partner[doublet]]       # two cells in one droplet
    counts = sample_negative_binomial(rng, mu, theta_g)
    obs = pd.DataFrame({
        "cell_type": np.where(doublet, "Doublet", np.array(names, dtype=object)[cell_type]),
        "true_type": np.array(names, dtype=object)[cell_type],
        "lineage": np.array([cell_types[n][0] for n in names], dtype=object)[cell_type],
        "donor": [f"donor{d + 1}" for d in donor],
        "condition": np.where(stimulated, "stimulated", "control"),
        "batch": [f"batch{b + 1}" for b in batch],
        "n_counts": counts.sum(axis=1),
        "n_genes": (counts > 0).sum(axis=1),
        "pct_mito": 100 * counts[:, is_mito].sum(axis=1) / np.maximum(counts.sum(axis=1), 1),
        "is_doublet": doublet, "is_damaged": damaged, "is_cycling": cycling,
    }, index=[f"cell{i:05d}" for i in range(n_cells)])
    var = pd.DataFrame({"marker_of": marker_of, "is_mito": is_mito, "is_interferon": is_ifn,
                        "is_cycling": is_cyc, "theta": theta_g}, index=genes)
    return dict(counts=counts, obs=obs, var=var, log_profiles=pd.DataFrame(profiles, index=names, columns=genes),
                true_logfc=pd.DataFrame(true_logfc, index=names, columns=genes))


IUPAC = {"A": "A", "C": "C", "G": "G", "T": "T", "R": "AG", "Y": "CT", "S": "CG", "W": "AT", "K": "GT",
         "M": "AC", "N": "ACGT"}


REGULATORY_MOTIFS = {
    "AP-1": ("TGASTCA", 1.0),
    "GATA": ("WGATAR", 0.7),
    "SP1": ("GGGGCGGGG", 0.5),
    "E-box": ("CACGTG", 0.8),
    "REST": ("TTCAGCACC", -1.5),        # repressor
}


def reverse_complement(seq: str) -> str:
    """Reverse complement of a DNA string (IUPAC codes are complemented too)."""
    comp = {"A": "T", "C": "G", "G": "C", "T": "A", "R": "Y", "Y": "R", "S": "S", "W": "W", "K": "M",
            "M": "K", "N": "N"}
    return "".join(comp[b] for b in reversed(seq.upper()))


def find_motif_sites(seq: str, consensus: str) -> list:
    """Start positions where ``consensus`` (IUPAC) matches ``seq`` on either strand (overlaps allowed)."""
    import re
    sites = set()
    for c in {consensus, reverse_complement(consensus)}:
        pattern = "(?=" + "".join(f"[{IUPAC[b]}]" for b in c) + ")"
        sites.update(m.start() for m in re.finditer(pattern, seq))
    return sorted(sites)


def regulatory_activity(seq: str) -> float:
    """Noise-free enhancer activity of a DNA sequence under the course's motif grammar.

    Activity = Σ motif effects (sites in the central half count fully, flanking sites 60 %)
    + 1.0 if an AP-1 and a GATA site lie within 30 bp of each other (cooperative binding)
    + 2·(GC fraction − 0.5).  This is the "true biology" that Module 36's networks must learn.
    """
    L = len(seq)
    total, starts = 0.0, {}
    for name, (consensus, effect) in REGULATORY_MOTIFS.items():
        starts[name] = find_motif_sites(seq, consensus)
        for s in starts[name]:
            total += effect * (1.0 if L / 4 <= s + len(consensus) / 2 <= 3 * L / 4 else 0.6)
    if any(abs(a - g) <= 30 for a in starts["AP-1"] for g in starts["GATA"]):
        total += 1.0
    gc = (seq.count("G") + seq.count("C")) / max(L, 1)
    return total + 2.0 * (gc - 0.5)


def simulate_regulatory_sequences(rng: np.random.Generator, n_seqs: int = 4000, length: int = 200,
                                  motif_rate: float = 1.5, noise: float = 0.3, n_chromosomes: int = 8) -> dict:
    """Simulate an enhancer reporter assay (MPRA / STARR-seq style): DNA sequence → activity.

    Background DNA has a per-sequence GC content; each sequence receives Poisson(``motif_rate``)
    planted motif instances at random positions and orientations (sampled from the IUPAC
    consensus).  The measured activity is ``regulatory_activity(seq)`` plus Gaussian noise,
    so motifs that occur by chance count too, exactly as in real DNA.  Each sequence is
    assigned to a chromosome so that models can be tested on held-out chromosomes.

    Returns a dict with ``sequences`` (list of str), ``activity``, ``label`` (1 = active, top
    40 %), ``chromosome`` and ``sites`` (DataFrame of planted motifs).
    """
    names = list(REGULATORY_MOTIFS)
    seqs, rows = [], []
    for i in range(n_seqs):
        gc = rng.beta(8, 8)
        s = list(rng.choice(list("ACGT"), length, p=[(1 - gc) / 2, gc / 2, gc / 2, (1 - gc) / 2]))
        for _ in range(rng.poisson(motif_rate)):
            m = names[rng.integers(len(names))]
            inst = "".join(rng.choice(list(IUPAC[b])) for b in REGULATORY_MOTIFS[m][0])
            strand = "+" if rng.random() < 0.5 else "-"
            if strand == "-":
                inst = reverse_complement(inst)
            start = int(rng.integers(0, length - len(inst)))
            s[start:start + len(inst)] = list(inst)
            rows.append((i, m, start, strand))
        seqs.append("".join(s))
    activity = np.array([regulatory_activity(s) for s in seqs]) + rng.normal(0, noise, n_seqs)
    label = (activity > np.quantile(activity, 0.6)).astype(int)
    chromosome = np.array([f"chr{c + 1}" for c in rng.integers(0, n_chromosomes, n_seqs)], dtype=object)
    sites = pd.DataFrame(rows, columns=["seq_id", "motif", "start", "strand"])
    return dict(sequences=seqs, activity=activity, label=label, chromosome=chromosome, sites=sites)


def simulate_helix_bundle(n_helices: int = 4, helix_length: int = 14, loop_length: int = 4,
                          rng=None, noise: float = 0.0) -> dict:
    """Cα coordinates (Å) of an idealised four-helix-bundle protein, its contact map and secondary structure.

    Helices are ideal right-handed α-helices (3.6 residues per turn, 1.5 Å rise, 2.3 Å radius)
    packed antiparallel on the corners of a 10 Å square and joined by short loops.  ``noise`` adds
    Gaussian jitter (needs ``rng``).  Contacts: Cα–Cα distance < 8 Å and |i − j| ≥ 3.

    Returns a dict with ``coords`` (L × 3), ``ss`` (string of 'H'/'L'), ``distances`` and ``contacts``.
    """
    corners = 5.0 * np.array([[1, 1], [1, -1], [-1, -1], [-1, 1]], float)
    pts, ss = [], ""
    for h in range(n_helices):
        cx, cy = corners[h % 4] * (1 + h // 4)
        up = h % 2 == 0
        k = np.arange(helix_length)
        angle = np.deg2rad(100.0) * k * (1 if up else -1)      # turn with the direction of travel, so
        z = 1.5 * (k - (helix_length - 1) / 2) * (1 if up else -1)   # every helix is right-handed
        helix = np.c_[cx + 2.3 * np.cos(angle), cy + 2.3 * np.sin(angle), z]
        if pts:                                                      # loop: circular arc, 3.8 Å steps
            start, end = pts[-1], helix[0]
            d, m = np.linalg.norm(end - start), loop_length + 1
            lo, hi = 1e-6, 2 * np.pi - 1e-6                              # arc angle Θ with m equal chords
            for _ in range(60):
                theta = (lo + hi) / 2
                lo, hi = (theta, hi) if 3.8 * np.sin(theta / 2) / np.sin(theta / (2 * m)) > d else (lo, theta)
            R = 1.9 / np.sin(theta / (2 * m))
            e1 = (end - start) / d
            e2 = np.r_[0, 0, np.sign(start[2])] - np.dot(np.r_[0, 0, np.sign(start[2])], e1) * e1
            e2 /= np.linalg.norm(e2)
            c = (start + end) / 2 - R * np.cos(theta / 2) * e2
            for k in range(1, m):
                psi = -theta / 2 + k * theta / m
                pts.append(c + R * (np.sin(psi) * e1 + np.cos(psi) * e2))
            ss += "L" * loop_length
        pts.extend(helix)
        ss += "H" * helix_length
    coords = np.array(pts)
    if noise > 0:
        coords = coords + rng.normal(0, noise, coords.shape)
    D = np.linalg.norm(coords[:, None] - coords[None], axis=-1)
    sep = np.abs(np.arange(len(coords))[:, None] - np.arange(len(coords))[None])
    return dict(coords=coords, ss=ss, distances=D, contacts=(D < 8.0) & (sep >= 3))


def potts_score(seqs, h, J) -> np.ndarray:
    """Statistical fitness Σ_i h_i(a_i) + Σ_{i<j} J_ij(a_i, a_j) of integer-coded sequences (higher = fitter)."""
    seqs = np.atleast_2d(seqs)
    L = seqs.shape[1]
    ii, jj = np.nonzero(np.triu(np.abs(J).sum(axis=(2, 3)) > 0, 1))      # coupled pairs only
    return h[np.arange(L), seqs].sum(axis=1) + J[ii, jj, seqs[:, ii], seqs[:, jj]].sum(axis=1)


def pca(X, n_components: int = 2):
    """Principal component analysis by SVD of the column-centred data.

    Returns ``(scores, components, explained_variance_ratio)``: scores are N × k, components k × J.
    """
    X = np.asarray(X, float)
    Xc = X - X.mean(axis=0)
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    var = S ** 2 / (len(X) - 1)
    return U[:, :n_components] * S[:n_components], Vt[:n_components], var[:n_components] / var.sum()


def benjamini_hochberg(pvalues) -> np.ndarray:
    """Benjamini–Hochberg adjusted p-values (q-values): reject where q ≤ α to control the FDR at α."""
    p = np.asarray(pvalues, float)
    order = np.argsort(p)
    ranked = p[order] * len(p) / np.arange(1, len(p) + 1)
    q = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty_like(p)
    out[order] = np.minimum(q, 1.0)
    return out


def plot_embedding(Z, labels, ax=None, title: str = "", s: float = 4, legend: bool = True, cmap=None):
    """Scatter a 2-D embedding (PCA, t-SNE, latent space, …) coloured by a categorical or numeric label."""
    Z = np.asarray(Z, float)
    labels = np.asarray(labels)
    ax = ax or plt.gca()
    if labels.dtype.kind in "fc":
        sc = ax.scatter(Z[:, 0], Z[:, 1], c=labels, s=s, cmap=cmap or "viridis", linewidths=0)
        plt.colorbar(sc, ax=ax, shrink=0.8)
    else:
        palette = plt.get_cmap("tab10" if len(np.unique(labels)) <= 10 else "tab20")
        for k, lab in enumerate(np.unique(labels)):
            m = labels == lab
            ax.scatter(Z[m, 0], Z[m, 1], s=s, color=palette(k % palette.N), label=str(lab), linewidths=0)
        if legend:
            ax.legend(markerscale=3, fontsize=7, frameon=False, loc="best")
    ax.set_xticks([]); ax.set_yticks([])
    if title:
        ax.set_title(title)
    return ax


print("🧰 Toolkit loaded:", ", ".join(['sample_negative_binomial', 'make_expression_programs', 'simulate_scrna', 'reverse_complement', 'find_motif_sites', 'regulatory_activity', 'simulate_regulatory_sequences', 'simulate_helix_bundle', 'potts_score', 'pca', 'benjamini_hochberg', 'plot_embedding']))

---
## 1 · Language modelling = next-token prediction

> 📍 **Where we are.** In [Module 19](19_hmm_inference.ipynb) a Markov chain generated a DNA sequence one letter at a time,
> each letter depending on the previous state. A **language model** is the same idea applied to words: a probability
> distribution over sequences, written one token at a time. Every LLM — from GPT-2 to today's chat assistants — is trained
> on exactly this task: predict the next token.

By the chain rule of probability ([Module 02](02_probability_for_biologists.ipynb)), any distribution over a sequence of
tokens $w_1,\dots,w_T$ can be written as

$$P(w_1,\dots,w_T)=\prod_{t=1}^{T}P(w_t\mid w_1,\dots,w_{t-1}).$$

*In words:* the probability of a sentence is the probability of its first word, times that of the second word given the
first, and so on. A language model is any model of the conditional distributions $P(w_t\mid w_{<t})$ — the "next-token
predictor". It can **score** text (how probable is this sentence?) and **generate** it (sample one token at a time).

**An n-gram model** truncates the history to the last $n-1$ tokens, $P(w_t\mid w_{<t})\approx P(w_t\mid w_{t-n+1},\dots,w_{t-1})$,
and estimates each conditional by counting, $\hat P(w\mid c)=\frac{\#(c,w)+k}{\#(c)+kV}$ ($V$ = vocabulary size, $k$ a small
pseudocount). *In words:* how often did $w$ follow the context $c$ in the training text, with a little probability reserved
for things never seen. It is a Markov chain of order $n-1$ whose transition table is learned by maximum likelihood.

> 🔗 **Builds on** [Module 20 · HMM learning](20_hmm_learning.ipynb), Section 1: when the states are observed, the MLE of a
> transition probability is a normalised count. An n-gram model is that estimator with words as states (Shannon 1951 used
> exactly such models to estimate the entropy of English).

**A SIMULATED language with a known truth.** Our corpus is a lab notebook. Each entry is generated by a small probabilistic
grammar from a hidden table of facts — which stimulus (interferon, LPS, anti-CD3) induces which genes in which of six blood
cell types (interferon acts on all of them, LPS only on monocytes and dendritic cells, anti-CD3 only on T cells; 5 % of
measurements are wrong). Because we wrote the grammar, we know the **true probability of every word** given its prefix.
Two cell–stimulus combinations — B cells with LPS and NK cells with anti-CD3 — are **never written down** in the notebook;
we will ask the model about them later.

In [ ]:
# ── 1.1 The SIMULATED world and its lab-notebook grammar ──
CELLS = ["monocytes", "dendritic_cells", "b_cells", "nk_cells", "cd4_t_cells", "cd8_t_cells"]
STIMULI = ["interferon", "lps", "anti_cd3"]
RESPONDERS = {"interferon": ["isg15", "mx1", "ifit1", "oas1"], "lps": ["tnf", "il1b", "il6", "cxcl8"],
              "anti_cd3": ["il2", "ifng", "cd69", "gzmb"]}
RESPONSIVE = {"interferon": set(CELLS), "lps": {"monocytes", "dendritic_cells"}, "anti_cd3": {"cd4_t_cells", "cd8_t_cells"}}
GENES = sum(RESPONDERS.values(), []) + ["actb", "gapdh"]
VERBS, TIMES, ASSAYS = ["treated", "stimulated", "incubated"], ["2", "6", "24"], ["qpcr", "flow", "rnaseq"]
HELD_OUT = {("b_cells", "lps"), ("nk_cells", "anti_cd3")}             # never written in the notebook


def notebook_entry(rng, held_out=HELD_OUT):
    """One 22-token entry and the TRUE probability of each token given everything before it."""
    toks, probs = [], []

    def emit(token, p):
        toks.append(token); probs.append(p)

    emit("<s>", 1.0); emit("day", 1.0); emit(str(rng.integers(1, 10)), 1 / 9); emit(":", 1.0); emit("we", 1.0)
    emit(VERBS[rng.integers(3)], 1 / 3)
    cell = CELLS[rng.integers(6)]; emit(cell, 1 / 6); emit("with", 1.0)
    allowed = [s for s in STIMULI if (cell, s) not in held_out]
    stim = allowed[rng.integers(len(allowed))]; emit(stim, 1 / len(allowed))
    emit("for", 1.0); emit(TIMES[rng.integers(3)], 1 / 3); emit("h", 1.0); emit("and", 1.0); emit("measured", 1.0)
    others = [g for g in GENES if g not in RESPONDERS[stim]]
    gene = RESPONDERS[stim][rng.integers(4)] if rng.random() < 0.75 else others[rng.integers(len(others))]
    emit(gene, 0.75 / 4 if gene in RESPONDERS[stim] else 0.25 / len(others))
    emit("by", 1.0); emit(ASSAYS[rng.integers(3)], 1 / 3); emit(";", 1.0); emit(gene, 1.0); emit("was", 1.0)
    truly_up = gene in RESPONDERS[stim] and cell in RESPONSIVE[stim]
    up = rng.random() < (0.95 if truly_up else 0.05)                   # 5 % of measurements are wrong
    emit("up" if up else "unchanged", 0.95 if up == truly_up else 0.05); emit(".", 1.0)
    return toks, probs


rng = np.random.default_rng(SEED + 1)
train_corpus = [notebook_entry(rng) for _ in range(4000)]
test_corpus = [notebook_entry(rng) for _ in range(500)]
VOCAB = sorted({t for s, _ in train_corpus + test_corpus for t in s})
STOI = {t: i for i, t in enumerate(VOCAB)}
for s, _ in train_corpus[:3]:
    print(" ".join(s[1:]))
oracle_ce = -np.mean([np.log(p) for _, ps in test_corpus for p in ps[1:]])
print(f"\nSIMULATED corpus: {len(train_corpus)} training and {len(test_corpus)} test entries, vocabulary V = {len(VOCAB)}")
print(f"TRUE cross-entropy of the test entries: {oracle_ce:.3f} nats per token (perplexity {np.exp(oracle_ce):.2f})")

🧮 **Math refresher — cross-entropy and perplexity.** A model $q$ is scored on held-out text by the average negative
log-probability it gives to the tokens that actually occur,

$$H(q)=-\frac1N\sum_{t=1}^{N}\log q(w_t\mid w_{<t}),\qquad \mathrm{PPL}(q)=e^{H(q)}.$$

*In words:* the **cross-entropy** $H$ is the average surprise (in nats) of the model at each real token — the very loss used
to train classifiers in Modules 09 and 22; the **perplexity** is the number of equally likely choices the model is
effectively hesitating between. A uniform guess over $V$ words has perplexity $V$; a perfect predictor of a deterministic
text has perplexity 1. No model can beat the true distribution on average: its cross-entropy (the **entropy** of the
language) is the floor. Our grammar is mostly deterministic ("day", ":", "we", …), so its true perplexity is low; the
interesting tokens are the few that carry information.

In [ ]:
# ── 1.2 n-gram language models: Markov chains of order n − 1, learned by counting ──
class NGramLM:
    """P(w | previous n − 1 tokens) by add-k smoothed counts."""

    def __init__(self, n, k=0.01):
        self.n, self.k, self.counts = n, k, defaultdict(Counter)

    def fit(self, sentences):
        for s in sentences:
            padded = ["<s>"] * (self.n - 2) + s                      # entries already start with one <s>
            for i in range(self.n - 1, len(padded)):
                self.counts[tuple(padded[i - self.n + 1:i])][padded[i]] += 1
        return self

    def log_probs(self, s):
        """log P of tokens 1 … 21 of one entry, each given its n − 1 predecessors."""
        padded, out = ["<s>"] * (self.n - 2) + s, []
        for i in range(self.n - 1, len(padded)):
            c = self.counts[tuple(padded[i - self.n + 1:i])]
            out.append(np.log((c[padded[i]] + self.k) / (sum(c.values()) + self.k * len(VOCAB))))
        return np.array(out)


ngrams = {n: NGramLM(n).fit([s for s, _ in train_corpus]) for n in (1, 2, 3, 5, 8)}
ngram_lp = {n: np.array([m.log_probs(s) for s, _ in test_corpus]) for n, m in ngrams.items()}   # 500 × 21
ladder = {"uniform": np.log(len(VOCAB))} | {f"{n}-gram": -lp.mean() for n, lp in ngram_lp.items()} | {"TRUE grammar": oracle_ce}
for name, ce in ladder.items():
    print(f"{name:13s} cross-entropy {ce:.3f} nats/token   perplexity {np.exp(ce):6.2f}")

🔍 **What just happened?** The ladder is the history of language modelling in miniature. A uniform guess has perplexity 54
(the vocabulary size). Word frequencies alone (1-gram) barely help (38.6); one word of context (2-gram) is a huge step
(2.16), because most of this grammar is fixed word order. Longer contexts keep improving (5-gram 1.73, 8-gram 1.71) towards
the true perplexity of 1.65 — but not all the way. Section 2 shows exactly *which* words the n-grams get wrong and
why a transformer can do better.

⚠️ **Why not simply use a 20-gram?** The number of possible contexts grows as $V^{n-1}$: most long contexts are never seen in
training, so their counts are zero and the model falls back on the pseudocount. This **data sparsity** is why Bengio
*et al.* (2003) proposed *neural* language models, which represent words by learned vectors and share statistical strength
between similar contexts.

✏️ **Your turn.** Generate five entries from the 2-gram model by sampling one token at a time from `ngrams[2].counts`.
Are they grammatical? Do they respect the facts (e.g. is "lps … tnf … up" ever written for T cells)?

---
## 2 · 🧪 A tiny GPT: a transformer that predicts the next word

> 📍 **Where we are.** [Module 26](26_attention_transformers.ipynb) built a transformer *encoder* that sees the whole
> sequence and fills in masked tokens. A **GPT** (generative pre-trained transformer) is a *decoder*: it must predict
> token $t$ from tokens $1,\dots,t-1$ only, so that it can generate text left to right. One change makes this possible —
> the **causal mask**.

**Causal self-attention.** In ordinary self-attention, position $i$ averages the values of all positions $j$ with weights
$A_{ij}=\operatorname{softmax}_j(\mathbf q_i\cdot\mathbf k_j/\sqrt{d_k})$. A decoder sets the scores of all *future*
positions to $-\infty$ before the softmax:

$$A_{ij}=\frac{\exp(s_{ij})\,[j\le i]}{\sum_{j'\le i}\exp(s_{ij'})},\qquad s_{ij}=\frac{\mathbf q_i\cdot\mathbf k_j}{\sqrt{d_k}}.$$

*In words:* each position may look back at itself and everything before it, never ahead; $[j\le i]$ is 1 when position $j$
is not in the future and 0 otherwise. Training then predicts every next token of a sequence in parallel, with the
cross-entropy loss of Section 1 — and at generation time the model can extend a prompt one token at a time.

> 🔗 **Builds on** [Module 26](26_attention_transformers.ipynb), Sections 2–3 (attention, transformer blocks) and
> [Module 22](22_neural_networks.ipynb) (training loops). Compared with an n-gram, the context is not truncated: attention
> can reach any earlier word, and words are represented by learned embeddings, so similar contexts share statistics.

In [ ]:
# ── 2.1 A decoder-only transformer ("TinyGPT") ──
class CausalSelfAttention(nn.Module):
    """Multi-head self-attention in which position i only attends to positions ≤ i."""

    def __init__(self, d_model, n_heads):
        super().__init__()
        self.h, self.qkv, self.out = n_heads, nn.Linear(d_model, 3 * d_model), nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, self.h, D // self.h).permute(2, 0, 3, 1, 4)
        scores = q @ k.transpose(-2, -1) / math.sqrt(D // self.h)
        future = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
        A = scores.masked_fill(future, float("-inf")).softmax(-1)          # the causal mask
        return self.out((A @ v).transpose(1, 2).reshape(B, T, D))


class GPTBlock(nn.Module):
    """Pre-LayerNorm transformer block: x + attention(LN(x)), then x + MLP(LN(x))."""

    def __init__(self, d_model, n_heads):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads)
        self.mlp = nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(), nn.Linear(4 * d_model, d_model))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))


class TinyGPT(nn.Module):
    """Token + position embeddings → 2 causal blocks → logits for the next token at every position."""

    def __init__(self, vocab_size, n_pos, d_model=64, n_heads=4, n_layers=2):
        super().__init__()
        self.tok, self.pos = nn.Embedding(vocab_size, d_model), nn.Embedding(n_pos, d_model)
        self.blocks = nn.ModuleList([GPTBlock(d_model, n_heads) for _ in range(n_layers)])
        self.ln, self.head = nn.LayerNorm(d_model), nn.Linear(d_model, vocab_size)

    def forward(self, idx):
        x = self.tok(idx) + self.pos.weight[: idx.shape[1]]
        for block in self.blocks:
            x = block(x)
        return self.head(self.ln(x))

In [ ]:
# ── 2.2 Train four model sizes on the same corpus (≈ 20 s in total) ──
X_train = torch.tensor([[STOI[t] for t in s] for s, _ in train_corpus])      # 4000 × 22 token ids
X_test = torch.tensor([[STOI[t] for t in s] for s, _ in test_corpus])


def train_gpt(d_model, steps=600, batch=32, lr=3e-3, seed=SEED):
    """Next-token training with AdamW and a one-cycle schedule; returns the model and its held-out cross-entropy."""
    torch.manual_seed(seed)
    model = TinyGPT(len(VOCAB), X_train.shape[1] - 1, d_model=d_model, n_heads=4 if d_model >= 16 else 2)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.1)
    for _ in range(steps):
        b = X_train[torch.randint(0, len(X_train), (batch,))]
        loss = F.cross_entropy(model(b[:, :-1]).reshape(-1, len(VOCAB)), b[:, 1:].reshape(-1))   # predict token t+1
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    model.eval()
    with torch.no_grad():
        ce = F.cross_entropy(model(X_test[:, :-1]).reshape(-1, len(VOCAB)), X_test[:, 1:].reshape(-1)).item()
    return model, ce


scaling = []
for d in (8, 16, 32, 64):
    t0 = time.time()
    model, ce = train_gpt(d)
    scaling.append((d, sum(p.numel() for p in model.parameters()), ce, time.time() - t0))
    print(f"d_model = {d:2d}: {scaling[-1][1]:7,d} parameters, held-out cross-entropy {ce:.3f} nats "
          f"(perplexity {np.exp(ce):.2f}), {scaling[-1][3]:.1f} s")
gpt = model                                                                   # keep the largest model
print(f"TRUE grammar: {oracle_ce:.3f} nats (perplexity {np.exp(oracle_ce):.2f})")

In [ ]:
# ── 2.3 🧪 Where is the information? Cross-entropy word slot by word slot ──
SLOTS = {"day": 2, "verb": 5, "cell": 6, "stimulus": 8, "time": 10, "gene": 14, "assay": 16, "gene (repeat)": 18, "result": 20}
true_lp = np.log(np.array([ps for _, ps in test_corpus]))                     # 500 × 22 TRUE log-probabilities
with torch.no_grad():
    gpt_lp_all = torch.log_softmax(gpt(X_test[:, :-1]), -1)                  # position t predicts token t + 1
gpt_lp = gpt_lp_all.gather(2, X_test[:, 1:, None])[..., 0].numpy()            # 500 × 21
slot_ce = pd.DataFrame({name: {"TRUE grammar": -true_lp[:, pos].mean(),
                               **{f"{n}-gram": -ngram_lp[n][:, pos - 1].mean() for n in (2, 3, 5, 8)},
                               "TinyGPT": -gpt_lp[:, pos - 1].mean()} for name, pos in SLOTS.items()})
print("cross-entropy (nats) at each slot of a notebook entry:")
print(slot_ce.round(3).to_string())

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.6), gridspec_kw={"width_ratios": [1, 1.4]})
names = ["2-gram", "3-gram", "5-gram", "8-gram", "TinyGPT"]
vals = [ladder[n] for n in names[:-1]] + [scaling[-1][2]]
a1.bar(range(len(names)), vals, color=[OKABE_ITO[5]] * 4 + [OKABE_ITO[0]])
a1.axhline(oracle_ce, color="k", ls="--", lw=1, label=f"TRUE grammar ({oracle_ce:.3f})")
a1.set_xticks(range(len(names))); a1.set_xticklabels(names); a1.set_ylim(0.4, 0.8)
a1.set_ylabel("held-out cross-entropy (nats/token)"); a1.legend(fontsize=7)
a1.set_title(f"SIMULATED lab notebook: the language-model ladder\n(uniform {ladder['uniform']:.2f}, "
             f"1-gram {ladder['1-gram']:.2f}: off the scale)")
hard = ["gene", "gene (repeat)", "result"]
for k, (row, c) in enumerate(zip(["3-gram", "5-gram", "8-gram", "TinyGPT"], [OKABE_ITO[3], OKABE_ITO[4], OKABE_ITO[5], OKABE_ITO[0]])):
    a2.bar(np.arange(3) + (k - 1.5) * 0.2, slot_ce.loc[row, hard] - slot_ce.loc["TRUE grammar", hard], 0.2, color=c, label=row)
a2.set_xticks(range(3)); a2.set_xticklabels(["gene\n(depends on the stimulus)", "gene repeated\n(copy 4 words back)",
                                             "result\n(depends on cell, stimulus, gene)"])
a2.set_ylabel("excess cross-entropy over the truth (nats)"); a2.legend(fontsize=7)
a2.set_title("which words need long-range context?")
plt.tight_layout(); plt.show()

🔍 **What just happened?** All models are essentially perfect on the fixed words and on the slots that are pure chance
(day, verb, cell, time, assay — their cross-entropy equals the truth, e.g. $\log 9=2.20$ for the day). The differences sit in
three slots, and each needs context of a different length:

* **gene** depends on the stimulus, 6 words earlier. Models with shorter context (2-, 3-, 5-gram) only know "…and measured"
  and spread probability over all genes (2.59 nats vs the true 2.16); the 8-gram (2.16) and TinyGPT (2.17) see the stimulus.
* **gene (repeat)** is a copy of the gene four words back: trivial for a 5-gram or the GPT (0.005 and 0.014 nats),
  impossible for a 3-gram (2.59). Copying earlier tokens is a classic skill of attention ("induction heads").
* **result** ("up"/"unchanged") depends on the *combination* of cell type, stimulus and gene — up to 14 words back. Every
  n-gram is stuck at 0.55–0.69 nats; TinyGPT reaches 0.18, close to the 0.16 of the true grammar on these test entries (the
  floor set by the 5 % measurement errors). It has learned the hidden fact table.

Overall TinyGPT reaches 0.509 nats per token against the truth's 0.502 — a perplexity of 1.66 vs 1.65 — with about
100,000 parameters and 600 training steps.

**Scaling.** The four model sizes trained in cell 2.2 let us look at the most famous empirical finding about LLMs: held-out
loss falls smoothly, roughly as a **power law**, as models (and data and compute) grow (Kaplan *et al.* 2020). With a known
floor (the entropy of the language) we can plot the *reducible* part of the loss.

In [ ]:
# ── 2.4 A mini scaling experiment: reducible loss vs number of parameters ──
params = np.array([s[1] for s in scaling]); reducible = np.array([s[2] for s in scaling]) - oracle_ce
slope, intercept = np.polyfit(np.log(params), np.log(reducible), 1)
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.loglog(params, reducible, "o-", color=OKABE_ITO[0], ms=8, label="TinyGPT, 600 steps each")
ax.loglog(params, np.exp(intercept) * params ** slope, "--", color=OKABE_ITO[1], label=f"power law, slope {slope:.2f}")
for (d, n, ce, _), r in zip(scaling, reducible):
    ax.annotate(f"d = {d}", (n, r), textcoords="offset points", xytext=(5, 5), fontsize=8)
ax.set_xlabel("number of parameters"); ax.set_ylabel("held-out loss − entropy of the language (nats)")
ax.set_title("SIMULATED corpus: bigger models approach the floor"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
print(f"reducible loss: {np.round(reducible, 4)}; fitted exponent {slope:.2f}")

🔍 **What just happened?** The reducible loss falls from 0.21 to 0.007 nats as the model grows from about 3,000 to 100,000
parameters, roughly along a straight line on log–log axes (slope −0.95). Real scaling laws are measured over many orders of
magnitude and also involve data and compute — Hoffmann *et al.* (2022) showed that, for a fixed compute budget, models
should be trained on many more tokens than had been common. Our four points only show the *qualitative* trend; with four
points and one corpus, the fitted exponent means little.

⚠️ **Lower loss is not the same as more useful.** Most of the remaining loss in a real corpus is irreducible (people write
differently), and the abilities that matter for science — getting facts right, admitting ignorance — are not measured by
perplexity.

**Sampling and temperature.** To generate text, the model's next-token probabilities are reshaped by a **temperature** $T$,
$p_T(w)\propto p(w)^{1/T}$ (the logits are divided by $T$, as in the calibration of [Module 42](42_trustworthy_ml.ipynb)),
and one token is drawn. Low $T$ makes the choice nearly deterministic; high $T$ flattens the distribution.

In [ ]:
# ── 2.5 Sampling with temperature: grammar and facts of generated entries ──
@torch.no_grad()
def generate(model, n, temperature, seed=SEED, prompt=("<s>",)):
    """Sample n entries of 22 tokens, one token at a time, at the given temperature."""
    g = torch.Generator().manual_seed(seed)
    x = torch.tensor([[STOI[t] for t in prompt]] * n)
    while x.shape[1] < X_train.shape[1]:
        logits = model(x)[:, -1] / temperature
        x = torch.cat([x, torch.multinomial(torch.softmax(logits, -1), 1, generator=g)], 1)
    return [[VOCAB[i] for i in row] for row in x.tolist()]


def check_entry(t):
    """(grammatical?, factually correct result?) for one generated entry, judged by the TRUE grammar and fact table."""
    template_ok = (t[1:5] == ["day", t[2], ":", "we"] and t[2] in list("123456789") and t[5] in VERBS and t[6] in CELLS
                   and t[7] == "with" and t[8] in STIMULI and t[9] == "for" and t[10] in TIMES and t[11:14] == ["h", "and", "measured"]
                   and t[14] in GENES and t[15] == "by" and t[16] in ASSAYS and t[17] == ";" and t[18] == t[14]
                   and t[19] == "was" and t[20] in ("up", "unchanged") and t[21] == ".")
    if not template_ok:
        return False, False
    truth = t[14] in RESPONDERS[t[8]] and t[6] in RESPONSIVE[t[8]]
    return True, (t[20] == "up") == truth


for T in (0.5, 1.0, 1.5, 2.0):
    samples = generate(gpt, 300, T)
    ok = np.array([check_entry(t) for t in samples])
    bad = [t for t, (g, f) in zip(samples, ok) if not f]
    print(f"T = {T}: grammatical {ok[:, 0].sum()}/300, grammatical AND factually correct {ok[:, 1].sum()}/300")
    if bad:
        print(f"         a rejected sample: '{' '.join(bad[0][1:])}'")

🔍 **What just happened?** At $T=0.5$ all 300 sampled entries are grammatical and all but one are factually correct: the
model nearly always picks its top choice. At $T=1$ (the model's own distribution) about 92 % are grammatical and 88 % also
correct — small probabilities assigned to wrong words add up over 21 tokens. At $T=1.5$ only a quarter are grammatical, at
$T=2$ none. Look at the rejected samples: "b_cells … anti_cd3 … cd69 was up" is perfectly fluent and false. The model is
reproducing, at a low rate, the 5 % of erroneous measurements in its training notebook — language models learn the errors in
their data along with the facts. Temperature
trades diversity for reliability; for factual tasks low temperatures are used.

**Hallucination.** Now the critical test. The notebook never mentions B cells with LPS or NK cells with anti-CD3. What does
the model "believe" happens? We prompt it with an entry up to "… was" and read its probability for "up". For combinations
it has seen, the truth is known to it; for the two held-out combinations the truth (in our world: no response) was never
in the data.

In [ ]:
# ── 2.6 🧪 Asking about experiments that were never written down ──
@torch.no_grad()
def p_next(model, tokens):
    """Probability distribution of the next token after a prompt (list of tokens)."""
    return torch.softmax(model(torch.tensor([[STOI[t] for t in tokens]]))[0, -1], -1).numpy()


def p_up(cell, stim, gene):
    prompt = ["<s>", "day", "3", ":", "we", "stimulated", cell, "with", stim, "for", "6", "h", "and", "measured", gene,
              "by", "qpcr", ";", gene, "was"]
    return p_next(gpt, prompt)[STOI["up"]]


combos = [("monocytes", "lps"), ("cd4_t_cells", "lps"), ("b_cells", "interferon"), ("cd8_t_cells", "anti_cd3"),
          ("monocytes", "anti_cd3"), ("b_cells", "lps"), ("nk_cells", "anti_cd3")]
rows = [(f"{c} + {s}", "HELD OUT" if (c, s) in HELD_OUT else "seen", 0.95 if c in RESPONSIVE[s] else 0.05,
         np.mean([p_up(c, s, g) for g in RESPONDERS[s]])) for c, s in combos]
probe = pd.DataFrame(rows, columns=["experiment", "in training text?", "TRUE P(up)", "TinyGPT P(up)"])
print(probe.round(2).to_string(index=False))
fig, ax = plt.subplots(figsize=(9, 3.2))
colors = [OKABE_ITO[1] if r == "HELD OUT" else OKABE_ITO[0] for r in probe["in training text?"]]
ax.bar(range(len(probe)), probe["TinyGPT P(up)"], color=colors)
ax.scatter(range(len(probe)), probe["TRUE P(up)"], marker="_", s=400, color="k", zorder=3, label="truth")
ax.set_xticks(range(len(probe))); ax.set_xticklabels(probe["experiment"], rotation=20, fontsize=8)
ax.set_ylabel("P('up') for the stimulus's responder genes"); ax.legend(fontsize=8)
ax.set_title("SIMULATED: seen experiments (blue) match the truth; unseen ones (orange) are guesses")
plt.tight_layout(); plt.show()

🔍 **What just happened?** For the five combinations present in the notebook, TinyGPT's probabilities sit close to the
truth (0.87–0.97 for real responses, 0.02–0.09 for non-responses). For the two experiments nobody wrote down it gives 0.06
for B cells + LPS — right, as it happens — and 0.56 for NK cells + anti-CD3 — wrong (in our world NK cells do not respond).
Neither number is knowledge; both are *analogies* from the recorded experiments. NK cells behave exactly like T cells in
everything the notebook records (up with interferon, unchanged with LPS), so the model guesses they also respond to anti-CD3
like T cells. Sampled at $T=1$, it would write "nk_cells … anti_cd3 … gzmb was up" in more than half of such entries — in
exactly the same fluent, confident form as its true statements — and nothing in the output tells you which of its two
guesses was the lucky one.

That is **hallucination**: generation that is not grounded in evidence. It is not a bug that more training removes; it is
what a model of $P(\text{next token})$ does when asked about something its training data do not determine (Ji *et al.* 2023
review the many forms). The fluency of the text carries no information about its truth. The remedy is to **ground**
answers in retrievable evidence (Sections 4–5), to let models consult tools and data (Section 6), and to verify.

✏️ **Your turn.** Remove ("monocytes", "lps") from the training corpus as well (add it to `HELD_OUT`, regenerate the corpus,
retrain the d = 64 model). Is P(up) for monocytes + LPS now closer to the truth than for B cells + LPS? What does that tell
you about which unseen facts a language model can and cannot "infer"?

### 📄 Research connection — Language models: from n-grams to LLMs

- **What the field does.** Shannon (1951) estimated the entropy of English with n-gram predictions. Bengio *et al.* (2003) introduced neural
    language models with learned word embeddings; word2vec (Mikolov *et al.* 2013) made embeddings cheap. The transformer
    (Vaswani *et al.* 2017) enabled GPT-style decoders; GPT-3 (Brown *et al.* 2020) showed that a very large model can
    perform new tasks from a few examples in the prompt. Scaling laws (Kaplan *et al.* 2020; Hoffmann *et al.* 2022) describe
    how loss falls with model size, data and compute; instruction tuning and reinforcement learning from human feedback
    (Ouyang *et al.* 2022) turned language models into assistants.
- **What this notebook builds.** Count-based n-grams, a 2-layer causal transformer with about 100,000 parameters, temperature sampling and a
    four-point scaling curve — all on a corpus whose **true** probabilities are known, so cross-entropy can be compared with
    the floor slot by slot. Real LLMs use the same objective with up to hundreds of billions of parameters and trillions of
    tokens, which cannot be checked against a true distribution.
- **Where the frontier is.** Open questions (as of 2025) include why some abilities seem to appear only at large scale, how to make models
    say "I don't know" (calibration and abstention), how much of benchmark performance reflects memorisation of
    contaminated test sets, and how to evaluate long-form scientific reasoning. Mechanistic interpretability studies circuits
    such as the induction heads that implement copying.
- **Read next.** Bengio *et al.* 2003, *A neural probabilistic language model* (JMLR); Brown *et al.* 2020, *Language models are
    few-shot learners* (NeurIPS); Kaplan *et al.* 2020, *Scaling laws for neural language models* (arXiv preprint).

---
## 3 · Tokenisation and the stages of training

> 📍 **Where we are.** Our toy language came with ready-made words. Real models must first decide what a "token" is — a
> character, a word, a piece of a word, a $k$-mer of DNA — and real assistants go through several training stages after
> next-token pre-training. Both choices shape what a model can learn.

**Byte-pair encoding (BPE)** (Sennrich, Haddow & Birch 2016) builds a vocabulary from data: start from single characters,
count all adjacent pairs, merge the most frequent pair into a new token, and repeat. Frequent words — or frequent DNA
motifs — become single tokens; rare strings stay split into pieces. Most LLMs use BPE or a close relative, and several
genomic language models use BPE on DNA ([Module 26](26_attention_transformers.ipynb), Section 8). We run it on SIMULATED
enhancers from `simulate_regulatory_sequences`, in which transcription-factor motifs are planted, and ask: **does BPE
rediscover the motifs?**

In [ ]:
# ── 3.1 Byte-pair encoding on SIMULATED regulatory DNA ──
def bpe_learn(seqs, n_merges):
    """Learn n_merges BPE merges; returns the merged tokens (in order) and the tokenised sequences."""
    corpus, merges = [list(s) for s in seqs], []
    for _ in range(n_merges):
        pairs = Counter()
        for toks in corpus:
            pairs.update(zip(toks, toks[1:]))
        (a, b), _ = pairs.most_common(1)[0]
        merges.append(a + b)
        new_corpus = []
        for toks in corpus:                                        # replace every occurrence of the pair (a, b)
            out, i = [], 0
            while i < len(toks):
                if i + 1 < len(toks) and toks[i] == a and toks[i + 1] == b:
                    out.append(a + b); i += 2
                else:
                    out.append(toks[i]); i += 1
            new_corpus.append(out)
        corpus = new_corpus
    return merges, corpus


enh = simulate_regulatory_sequences(np.random.default_rng(SEED + 31), n_seqs=400, length=200, motif_rate=4.0)
merges, tokenised = bpe_learn(enh["sequences"], 60)
instances = set()                                                  # every DNA string that matches a planted motif
for consensus, _ in REGULATORY_MOTIFS.values():
    for c in (consensus, reverse_complement(consensus)):
        instances.update("".join(p) for p in product(*[IUPAC[b] for b in c]))
in_motif = lambda tok: any(tok in m for m in instances)
long_tokens = [t for t in merges if len(t) >= 4]
for k in (4, 5):
    kmers = ["".join(p) for p in product("ACGT", repeat=k)]
    learned = [t for t in long_tokens if len(t) == k]
    print(f"{k}-letter tokens learned: {learned}")
    print(f"   🧪 inside a planted motif: {np.mean([in_motif(t) for t in learned]):.2f} of them; "
          f"for a random {k}-mer: {np.mean([in_motif(t) for t in kmers]):.2f}")
print(f"SIMULATED enhancers: tokens per 200-bp sequence: characters 200, BPE (60 merges) {np.mean([len(t) for t in tokenised]):.0f}")

🔍 **What just happened?** Without being told anything about biology, BPE turned pieces of the planted motifs into tokens:
the four 5-letter tokens are CACGT (from the E-box CACGTG), CAGCA (from the REST site TTCAGCACC), and CGGGG and GCCCC (from
the GC-rich SP1 site GGGGCGGGG and its reverse complement); the 4-letter tokens include GATA (the GATA motif) and GAGT/GACT
(from AP-1, TGA[C/G]TCA). 🧪 All four 5-letter tokens lie inside a planted motif, against 4 % of random 5-mers (for 4-letter
tokens: 38 % against 17 %). Frequent substrings become tokens, and in regulatory DNA the most frequent non-random substrings
are binding sites. Sequences also became 2.6 times shorter (77 tokens instead of 200 letters), which matters because attention costs grow
with the square of the length. The flip side: a tokeniser fixes the units the model can "see" — a variant that splits a
token changes the input in an all-or-nothing way, one reason why many protein and DNA models still use single residues.

**From a next-token predictor to an assistant: the training stages.**

| Stage | Data | What changes | Example |
|---|---|---|---|
| **Pre-training** | trillions of tokens of text (and code) | learns $P(\text{next token})$: grammar, facts, styles | GPT-3 (Brown *et al.* 2020) |
| **Supervised fine-tuning** ("instruction tuning") | tens of thousands of (instruction, good answer) pairs written by people | learns to follow instructions and answer in a helpful format | — |
| **Preference optimisation** | pairs of answers ranked by people | shifts probability towards preferred answers (helpful, honest, harmless) | InstructGPT, RLHF (Ouyang *et al.* 2022); DPO (Rafailov *et al.* 2023) |
| **Prompting at use time** | the user's prompt (instructions, examples, retrieved documents) | no weights change; behaviour is steered by context | few-shot prompts; chain-of-thought (Wei *et al.* 2022) |

🧮 **Math refresher — learning from preferences.** RLHF first fits a **reward model** $r_\phi(x,y)$ that scores an answer
$y$ to prompt $x$, from human comparisons "answer $y_w$ is better than $y_l$", with the Bradley–Terry model

$$P(y_w\succ y_l\mid x)=\sigma\big(r_\phi(x,y_w)-r_\phi(x,y_l)\big).$$

*In words:* the probability that people prefer $y_w$ is a logistic function ([Module 09](09_logistic_regression.ipynb)) of
the difference in reward — exactly logistic regression on pairs. The language model is then tuned to produce high-reward
answers while staying close to its pre-trained distribution; DPO achieves the same goal with a single classification-like
loss on the preference pairs. Note what is optimised: answers that *people prefer*. Preferred answers are usually helpful
and correct, but a confident, well-written wrong answer can be preferred too — one reason why fine-tuned assistants still
hallucinate.

✏️ **Your turn.** Run `bpe_learn` on the *lab-notebook* corpus written as one long character string (join the entries with
spaces). After how many merges do whole words such as "measured" become single tokens? Which words never do, and why?

---
## 4 · 🧪 Embeddings and retrieval: finding the right documents

> 📍 **Where we are.** Section 2 showed that a model's *parametric* memory can be wrong exactly where data were missing.
> The alternative is to look things up. Retrieval is the first half of every literature assistant: given a question, find
> the most relevant documents. We build a small "PubMed" of 200 SIMULATED abstracts with **known topics and known facts**,
> so that retrieval quality can be measured exactly.

The synthetic abstracts cover eight topics: interferon responses, LPS and inflammation, T-cell activation, cell-type
markers, batch integration, CRISPR screens, protein structure and statistical genetics. The first three state facts drawn
from the same hidden biology as `simulate_scrna`: interferon induces interferon-stimulated genes (ISG15, MX1, IFIT1, …) in
every blood cell type — in our synthetic literature only 7 of the 10 interferon genes of the simulator are ever mentioned —
LPS acts on monocytes and dendritic cells, anti-CD3 on T cells. Abstracts also contain generic sentences and sentences
borrowed from other topics (e.g. "cells were annotated as monocytes using CD14 and LYZ"), as real abstracts do.

In [ ]:
# ── 4.1 A SIMULATED literature: 200 abstracts with known topics and known facts ──
CELL_TEXT = {"CD4 T": "CD4 T cells", "CD8 T": "CD8 T cells", "NK": "NK cells", "B": "B cells", "Monocyte": "monocytes",
             "Dendritic": "dendritic cells"}
MARKERS = {t: v[1] for t, v in PBMC_TYPES.items()}
LIT_GENES = {"interferon": ["ISG15", "MX1", "IFIT1", "IFIT3", "OAS1", "IRF7", "STAT1"],      # 7 of the 10 simulator ISGs
             "LPS": ["TNF", "IL1B", "IL6", "CXCL8"], "anti-CD3": ["IL2", "IFNG", "CD69", "GZMB"]}
LIT_RESPONSIVE = {"interferon": list(CELL_TEXT), "LPS": ["Monocyte", "Dendritic"], "anti-CD3": ["CD4 T", "CD8 T"]}
TOPICS = ["interferon response", "LPS and inflammation", "T-cell activation", "cell-type markers", "batch integration",
          "CRISPR screens", "protein structure", "statistical genetics"]
TOPIC_STIM = {"interferon response": "interferon", "LPS and inflammation": "LPS", "T-cell activation": "anti-CD3"}
METHOD_SENTENCES = {
    "batch integration": ["Batch effects between sequencing runs obscured biological variation.",
                          "We compare integration methods that align cells across batches.",
                          "Over-correction removed genuine differences between conditions.",
                          "Mixing metrics and conservation of cell types were benchmarked."],
    "CRISPR screens": ["Pooled CRISPR screens with single-cell readout measured perturbation effects.",
                       "Guide RNAs targeting transcription factors were delivered to cells.",
                       "Knockdown efficiency varied between guides.", "Perturbation effects revealed a regulatory network."],
    "protein structure": ["Protein language models predict residue contacts from sequence.",
                          "Multiple sequence alignments capture coevolution between residues.",
                          "Predicted structures were compared with experimental structures.",
                          "Variant effects were scored with a masked language model."],
    "statistical genetics": ["Genome-wide association studies identified loci for immune traits.",
                             "Fine-mapping prioritised candidate causal variants.",
                             "Heritability was estimated with a linear mixed model.",
                             "Polygenic scores were evaluated across ancestries."]}
GENERIC = ["Here we profiled {n} samples from healthy donors.", "Our results provide a resource for the community.",
           "Data and code are publicly available.", "We validated the main findings in an independent cohort.",
           "These results have implications for human disease.", "We discuss limitations and future directions."]
cap = lambda s: s[0].upper() + s[1:]


def make_abstract(rng, topic, i):
    """Title + 4–7 sentences; returns the facts (stimulus, cell type, gene) that the abstract asserts."""
    sents, facts = [], []
    if topic in TOPIC_STIM:
        stim = TOPIC_STIM[topic]
        title = [f"Single-cell responses to {stim}", f"Cell-type-specific {stim} responses in human blood"][rng.integers(2)]
        for _ in range(rng.integers(2, 4)):
            ct = LIT_RESPONSIVE[stim][rng.integers(len(LIT_RESPONSIVE[stim]))]
            g1, g2 = rng.choice(LIT_GENES[stim], 2, replace=False)
            sents.append([f"{cap(stim)} stimulation strongly induced {g1} and {g2} in {CELL_TEXT[ct]}.",
                          f"In {CELL_TEXT[ct]}, {g1} and {g2} were induced within hours of {stim} treatment."][rng.integers(2)])
            facts += [(stim, ct, g1), (stim, ct, g2)]
        unresponsive = [c for c in CELL_TEXT if c not in LIT_RESPONSIVE[stim]]
        if unresponsive and rng.random() < 0.7:
            sents.append(f"{cap(CELL_TEXT[unresponsive[rng.integers(len(unresponsive))]])} did not respond to {stim}.")
    elif topic == "cell-type markers":
        title = ["A marker atlas of human blood cells", "Annotating immune cell types"][rng.integers(2)]
        for _ in range(3):
            ct = list(MARKERS)[rng.integers(6)]; g1, g2 = rng.choice(MARKERS[ct], 2, replace=False)
            sents.append([f"{g1} and {g2} mark {CELL_TEXT[ct]}.",
                          f"{cap(CELL_TEXT[ct])} express high levels of {g1} and {g2}."][rng.integers(2)])
    else:
        title = {"batch integration": "Integrating single-cell data across batches",
                 "CRISPR screens": "Perturbation screens with single-cell readout",
                 "protein structure": "Learning protein structure from sequence",
                 "statistical genetics": "Genetic architecture of immune traits"}[topic]
        sents += list(rng.choice(METHOD_SENTENCES[topic], 3, replace=False))
    if rng.random() < 0.5:                                          # a borrowed annotation sentence (cross-topic noise)
        ct = list(MARKERS)[rng.integers(6)]; g1, g2 = rng.choice(MARKERS[ct], 2, replace=False)
        sents.append(f"Cells were annotated as {CELL_TEXT[ct]} using {g1} and {g2}.")
    sents += [s.format(n=rng.integers(4, 40)) for s in rng.choice(GENERIC, 2, replace=False)]
    sents = [sents[k] for k in rng.permutation(len(sents))]
    return dict(id=f"SIM-{1000 + i}", topic=topic, title=title, sentences=sents, facts=facts)


rng = np.random.default_rng(SEED + 41)
library = [make_abstract(rng, TOPICS[i % 8], i) for i in range(200)]
for doc in library[:2]:
    print(f"[{doc['id']}] ({doc['topic']}) {doc['title']}. {' '.join(doc['sentences'])}\n")
print(f"SIMULATED library: {len(library)} abstracts, {sum(len(d['facts']) for d in library)} stated (stimulus, cell, gene) facts")

**TF-IDF.** The classic way to represent a document for search is a vector with one entry per word of the vocabulary:

$$\text{tf-idf}(w,d)=\operatorname{tf}(w,d)\times\Big(\log\frac{N}{\operatorname{df}(w)}+1\Big),\qquad
\cos(\mathbf a,\mathbf b)=\frac{\mathbf a\cdot\mathbf b}{\|\mathbf a\|\,\|\mathbf b\|}.$$

*In words:* a word counts more the more often it appears in the document (term frequency, tf) and the *rarer* it is across
the $N$ documents (inverse document frequency; df = number of documents containing it) — "interferon" is informative,
"results" is not (Sparck Jones 1972). A query is turned into a vector the same way, and documents are ranked by the cosine of
the angle between vectors. **Latent semantic analysis** (LSA; Deerwester *et al.* 1990) compresses the document × word
matrix with a truncated SVD — the PCA of [Module 14](14_pca_dimensionality_reduction.ipynb) — so that words that co-occur
end up close together: a first, linear version of the **embeddings** that neural models learn.

In [ ]:
# ── 4.2 A search engine: tokeniser, TF-IDF, cosine ranking, LSA embeddings ──
STOPWORDS = set("a an the of in and or to for with by on at from as is are was were be this that these which what does do "
                "using within than into how our we here".split())


def tokenize(text):
    """Lower-case words of letters and digits, without stop words (so 'IFN-beta' → ['ifn', 'beta'])."""
    return [w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOPWORDS]


doc_tokens = [tokenize(d["title"] + " " + " ".join(d["sentences"])) for d in library]
WORDS = sorted({w for d in doc_tokens for w in d}); WIDX = {w: i for i, w in enumerate(WORDS)}
TF = np.zeros((len(library), len(WORDS)))
for i, toks in enumerate(doc_tokens):
    for w in toks:
        TF[i, WIDX[w]] += 1
IDF = np.log(len(library) / (TF > 0).sum(0)) + 1
DOC_TFIDF = TF * IDF
DOC_TFIDF /= np.linalg.norm(DOC_TFIDF, axis=1, keepdims=True)
U_lsa, S_lsa, Vt_lsa = np.linalg.svd(DOC_TFIDF, full_matrices=False)
K_LSA = 12
DOC_LSA = U_lsa[:, :K_LSA] * S_lsa[:K_LSA]
DOC_LSA /= np.linalg.norm(DOC_LSA, axis=1, keepdims=True)


def query_vector(text, weighting="tfidf"):
    """A query as a unit vector in word space (raw counts or TF-IDF) or in the 12-dimensional LSA space."""
    v = np.zeros(len(WORDS))
    for w in tokenize(text):
        if w in WIDX:
            v[WIDX[w]] += 1
    if weighting != "counts":
        v = v * IDF
    if weighting == "lsa":
        v = v @ Vt_lsa[:K_LSA].T
    n = np.linalg.norm(v)
    return v / n if n > 0 else v


def search(text, k=5, weighting="tfidf"):
    """Indices of the k best-matching abstracts and their cosine similarities."""
    M = {"counts": TF / np.linalg.norm(TF, axis=1, keepdims=True), "tfidf": DOC_TFIDF, "lsa": DOC_LSA}[weighting]
    sims = M @ query_vector(text, weighting)
    top = np.argsort(-sims)[:k]
    return top, sims[top]


print(f"vocabulary: {len(WORDS)} words; LSA keeps {K_LSA} dimensions "
      f"({(S_lsa[:K_LSA] ** 2).sum() / (S_lsa ** 2).sum():.0%} of the squared norm)")
top, sims = search("which genes respond to interferon in monocytes")
for i, s in zip(top, sims):
    print(f"  {s:.2f}  [{library[i]['id']}] ({library[i]['topic']}) {library[i]['title']}")

In [ ]:
# ── 4.3 🧪 Retrieval quality against the known topics: precision@k ──
QUERIES = {"interferon response": ["interferon stimulated genes", "antiviral interferon signalling in immune cells", "ISG15 and MX1 induction"],
           "LPS and inflammation": ["LPS inflammatory cytokines", "TNF and IL1B response of monocytes", "innate immune activation by bacterial LPS"],
           "T-cell activation": ["anti-CD3 T cell activation", "IFNG and GZMB in activated CD8 T cells", "IL2 production after T cell receptor stimulation"],
           "cell-type markers": ["marker genes for blood cell types", "which genes mark monocytes", "annotation of immune cell populations"],
           "batch integration": ["batch correction and integration", "removing technical variation between sequencing runs", "benchmark of data integration"],
           "CRISPR screens": ["CRISPR perturbation screen", "guide RNA knockdown with single-cell readout", "regulatory network from perturbations"],
           "protein structure": ["protein contact prediction", "language model of protein sequences", "coevolution and structure"],
           "statistical genetics": ["GWAS fine-mapping", "heritability of immune traits", "polygenic scores across ancestries"]}
topics_arr = np.array([d["topic"] for d in library])
rows = []
for weighting in ("counts", "tfidf", "lsa"):
    p5, p10 = [], []
    for topic, qs in QUERIES.items():
        for q in qs:
            top, _ = search(q, k=10, weighting=weighting)
            p5.append((topics_arr[top[:5]] == topic).mean()); p10.append((topics_arr[top] == topic).mean())
    rows.append((weighting, np.mean(p5), np.mean(p10), np.mean(np.array(p5) < 1)))
print(pd.DataFrame(rows, columns=["representation", "precision@5", "precision@10", "queries with a miss in the top 5"])
      .round(3).to_string(index=False))
fig, ax = plt.subplots(figsize=(7, 4))
plot_embedding(DOC_LSA[:, :2], topics_arr, ax=ax, s=14, title="SIMULATED abstracts in the first two LSA dimensions")
plt.tight_layout(); plt.show()

🔍 **What just happened?** On these keyword-rich queries retrieval mostly finds the right topic: precision@5 is 0.90 with raw
word counts and 0.96 with TF-IDF or LSA, which down-weight words that appear everywhere ("cells", "genes"). With raw counts 4
of the 24 queries have a wrong abstract in their top 5; with TF-IDF only 1, typically because a query word (such as
"monocytes") is shared between topics. The LSA map shows the "embedding" idea: abstracts on the same topic cluster
together, and related topics (the three stimulus topics, which share genes and cell types) sit near each other.

⚠️ **The weakness of exact words.** Keyword retrieval only matches words that literally occur. A question that says
"IFN-β" or "CD14⁺ cells" shares no words with abstracts that say "interferon" and "monocytes". Modern systems use dense
neural embeddings (trained so that paraphrases are close) or hybrid keyword + embedding search, and BM25 (Robertson &
Zaragoza 2009), a refined TF-IDF, remains a strong baseline. Section 5 shows what a vocabulary mismatch does downstream.

✏️ **Your turn.** Implement BM25 scoring, $\sum_{w\in q}\mathrm{idf}(w)\,\frac{\mathrm{tf}(w,d)(k_1+1)}{\mathrm{tf}(w,d)+k_1(1-b+b\,|d|/\overline{|d|})}$
with $k_1=1.5$, $b=0.75$, and compare its precision@5 with TF-IDF on `QUERIES`.

---
## 5 · Retrieval-augmented generation: answers with citations — and their failure modes

> 📍 **Where we are.** **Retrieval-augmented generation** (RAG; Lewis *et al.* 2020) puts retrieved documents into the
> model's context and asks it to answer from them, with citations. Literature assistants such as PaperQA (Lála *et al.*
> 2023, preprint) follow this recipe. We build the generator as a transparent **template composer** (no neural model), so
> that every step can be inspected.

Our composer receives a question and the top-$k$ abstracts and writes "Genes induced by S in C: G₁ [doc], G₂ [doc], …". We
compare two versions:

* a **naive composer**, which — like a fluent but careless model — lists the genes mentioned in the retrieved abstracts;
* a **verified composer**, which keeps a claim "G is induced by S in C" only if some retrieved sentence contains the gene,
  the stimulus, the cell type and an induction verb, and otherwise **abstains**.

In [ ]:
# ── 5.1 A retrieval-augmented answer composer, naive and verified ──
STIM_TERMS = {"interferon": ["interferon"], "LPS": ["lps"], "anti-CD3": ["anti-cd3", "anti cd3"]}
CELL_TERMS = {t: [CELL_TEXT[t].lower()] for t in CELL_TEXT}
ALL_GENES = sorted({g for gs in LIT_GENES.values() for g in gs} | {g for gs in MARKERS.values() for g in gs})


def find_terms(text, table):
    """Keys of `table` whose terms occur in the text (lower case)."""
    return [key for key, terms in table.items() if any(t in text.lower() for t in terms)]


def compose_answer(question, k=5, verify=False):
    """Retrieve k abstracts and compose a cited answer; with verify=True keep only claims supported by one sentence."""
    top, _ = search(question, k=k)
    stims, cellz = find_terms(question, STIM_TERMS), find_terms(question, CELL_TERMS)
    claims = {}
    for i in top:
        for sent in library[i]["sentences"]:
            for g in ALL_GENES:
                if re.search(rf"\b{g}\b", sent) is None:
                    continue
                supported = (bool(stims) and bool(cellz) and "induc" in sent.lower()
                             and all(find_terms(sent, STIM_TERMS).count(s) for s in stims)
                             and all(find_terms(sent, CELL_TERMS).count(c) for c in cellz))
                if supported or not verify:
                    claims.setdefault(g, set()).add(library[i]["id"])
    retrieved = ", ".join(f"{library[i]['id']} ({library[i]['topic']})" for i in top)
    if not claims:
        return "I cannot answer from the retrieved documents.", retrieved, set()
    answer = "; ".join(f"{g} [{', '.join(sorted(ids))}]" for g, ids in sorted(claims.items()))
    return f"Induced genes: {answer}.", retrieved, set(claims)


q1 = "Which genes are induced by interferon in monocytes?"
truth_q1 = {g for d in library for (s, c, g) in d["facts"] if s == "interferon" and c == "Monocyte"}
for verify in (False, True):
    answer, retrieved, genes = compose_answer(q1, verify=verify)
    print(f"{'VERIFIED' if verify else 'NAIVE'} composer: {answer}")
    print(f"   🧪 correct claims {len(genes & truth_q1)}/{len(genes)}; facts stated somewhere in the library: {sorted(truth_q1)}")
print("retrieved:", retrieved)

🔍 **What just happened?** For a well-phrased question retrieval works (all five abstracts are on interferon responses), and
both composers cite real abstracts. But the naive composer also "answers" with CD4, GNLY and PRF1 — marker genes that appear in
the same abstracts in *annotation* sentences: 3 of its 10 claims are false, because it confuses "mentioned near the topic"
with "stated as an answer". The verified composer keeps only claims that a single sentence supports: 7 of 7 are correct, each
with checkable citations, and here they cover all 7 interferon genes the library ever states for monocytes. (With other
questions it can miss facts that sit in abstracts outside the top 5.)

Now a question phrased the way a person might naturally ask it, with a synonym and a surface marker instead of the words
the abstracts use.

In [ ]:
# ── 5.2 A retrieval miss: synonyms → wrong documents → a confident wrong answer; then verification and query rewriting ──
q2 = "Which genes does IFN-beta induce in CD14+ cells?"
SYNONYMS = {"ifn-beta": "interferon", "ifn-b": "interferon", "type i ifn": "interferon", "cd14+ cells": "monocytes"}


def rewrite_query(question):
    """Replace known synonyms by the vocabulary of the library (a stand-in for query expansion)."""
    q = question.lower()
    for a, b in SYNONYMS.items():
        q = q.replace(a, b)
    return q


for label, q in [("original question", q2), ("rewritten question", rewrite_query(q2))]:
    print(f"── {label}: {q!r}  (query words: {tokenize(q)})")
    for verify in (False, True):
        answer, retrieved, genes = compose_answer(q, verify=verify)
        print(f"   {'VERIFIED' if verify else 'NAIVE   '}: {answer[:150]}{'…' if len(answer) > 150 else ''}")
    print(f"   retrieved: {retrieved[:160]}…")

🔍 **What just happened?** The original question shares no informative words with the interferon abstracts — the library
says "interferon" and "monocytes", the question says "IFN-beta" and "CD14+ cells". Retrieval therefore returns the abstracts
that do contain "cd14": the **marker** papers. The naive composer then produces a fluent, cited, **confidently wrong**
answer — "induced genes: CD14, CD19, CD3D, CD3E, …" — every citation real, every claim false. This is how retrieval-augmented systems
hallucinate: the generator is faithful to the wrong documents. The verified composer abstains because no retrieved sentence
links any gene to the stimulus. After rewriting the query into the library's vocabulary, retrieval finds the interferon
abstracts and the verified answer is correct again.

**RAG reduces hallucination but does not remove it.** Errors enter at every step: the query (vocabulary mismatch), the
retriever (wrong or incomplete documents), the documents themselves (outdated, contradicted, low quality, retracted), and the
generator (misreading, mixing up sentences, citing a paper for a claim it does not make). Practical defences: claim-level
verification against the cited text, abstention when support is missing, showing the retrieved passages to the reader, and
human spot checks.

✏️ **Your turn.** Ask `compose_answer("Which genes are induced by LPS in B cells?", verify=True)`. The library states that
B cells did *not* respond to LPS. Does the verified composer say so? Extend it to report negative statements ("did not
respond") as well.

---
## 6 · 🧪 Agents: a language model that acts

> 📍 **Where we are.** A retrieval system answers from documents. An **agent** goes further: it decides which tool to
> use, calls it, reads the result and decides what to do next — search, run an analysis, look something up — until it can
> answer. We build the loop with a **scripted policy** in place of the LLM, run it on a real analysis question, check the
> answer against the simulator's truth, and then break it on purpose.

**The ReAct loop** (Yao *et al.* 2023) interleaves *reasoning* and *acting*:

```text
Question → Thought → Action(tool, arguments) → Observation → Thought → Action → Observation → … → Final answer
```

At each step an LLM receives the question, the tool descriptions and everything so far, and outputs the next thought and
action (Toolformer, Schick *et al.* 2023, taught models to insert tool calls themselves). Our policy is a dozen `if`
statements that produce the same kind of output; the loop, the tools, the logs and the guardrails are exactly those of a
real agent.

**The question.** *"Which genes respond to interferon in monocytes, and is there literature support?"* Our agent's
laboratory has a SIMULATED single-cell experiment (`simulate_scrna`: 10 donors, half stimulated with interferon), whose true
response genes are known: the 10 interferon genes respond in every cell type and 40 further genes respond only in monocytes.

In [ ]:
# ── 6.1 The laboratory data and three tools ──
lab = simulate_scrna(np.random.default_rng(SEED + 61), n_cells=4000, n_genes=300, n_donors=10)
TYPE_NAMES = list(PBMC_TYPES)


def tool_lookup_marker(cell_type):
    """Marker genes of a cell type (from the course's PBMC table)."""
    if cell_type not in PBMC_TYPES:
        return dict(ok=False, text=f"ERROR: unknown cell type {cell_type!r}. Valid names: {TYPE_NAMES}")
    return dict(ok=True, text=f"{cell_type} markers: {', '.join(PBMC_TYPES[cell_type][1])}", data=PBMC_TYPES[cell_type][1])


def tool_differential_expression(cell_type, stimulus, data=lab, q_max=0.05, min_log2fc=1.0):
    """Pseudobulk DE (sum counts per donor, log-CPM, Welch t-test, BH) of stimulated vs control donors in one cell type.
    Guardrails: valid names, enough donors per condition, condition not confounded with batch."""
    if cell_type not in PBMC_TYPES:
        return dict(ok=False, text=f"ERROR: unknown cell type {cell_type!r}. Valid names: {TYPE_NAMES}")
    if stimulus != "interferon":
        return dict(ok=False, text=f"ERROR: no data for stimulus {stimulus!r} (this experiment used interferon only)")
    obs = data["obs"]
    donor, cond, batch = (obs[c].to_numpy(str) for c in ("donor", "condition", "batch"))
    donors = sorted(set(donor))
    stim_d = np.array([cond[donor == d][0] == "stimulated" for d in donors])
    if min(stim_d.sum(), (~stim_d).sum()) < 3:
        return dict(ok=False, text="REFUSED: fewer than 3 donors per condition — not enough replicates")
    if len({(c, b) for c, b in zip(cond, batch)}) == len(set(cond)) and len(set(batch)) > 1:
        return dict(ok=False, text="REFUSED: condition is completely confounded with batch — effects cannot be separated")
    in_type = obs["cell_type"].to_numpy(str) == cell_type
    pseudo = np.array([data["counts"][in_type & (donor == d)].sum(0) for d in donors], float)
    logcpm = np.log2(1 + 1e6 * pseudo / pseudo.sum(1, keepdims=True))
    log2fc = logcpm[stim_d].mean(0) - logcpm[~stim_d].mean(0)
    p = np.nan_to_num(stats.ttest_ind(logcpm[stim_d], logcpm[~stim_d], equal_var=False).pvalue, nan=1.0)
    table = pd.DataFrame({"gene": data["var"].index.to_numpy(str), "log2FC": log2fc, "q": benjamini_hochberg(p)})
    hits = table[(table.q < q_max) & (table.log2FC.abs() >= min_log2fc)].sort_values("q")
    return dict(ok=True, data=hits, text=f"{len(hits)} genes with q < {q_max} and |log2FC| ≥ {min_log2fc} "
                                         f"({stim_d.sum()} vs {(~stim_d).sum()} donors); top: {', '.join(hits.gene[:8])}")


def tool_search_literature(query, k=8):
    """TF-IDF search over the SIMULATED library (Section 4)."""
    top, sims = search(query, k=k)
    return dict(ok=True, data=[library[i] for i in top],
                text="; ".join(f"{library[i]['id']} '{library[i]['title']}' ({s:.2f})" for i, s in zip(top, sims)))


TOOLS = {"lookup_marker": tool_lookup_marker, "run_differential_expression": tool_differential_expression,
         "search_literature": tool_search_literature}
print("SIMULATED lab data:", lab["counts"].shape[0], "cells,", lab["obs"]["donor"].nunique(), "donors |",
      tool_differential_expression("Monocyte", "interferon")["text"])

In [ ]:
# ── 6.2 The agent loop (with a step limit and an approval gate) and a scripted policy ──
def run_agent(question, policy, tools=TOOLS, max_steps=6, needs_approval=(), approve=lambda name, args: False, show=True):
    """ReAct loop: ask the policy for (thought, action) or a final answer, execute tools, record observations."""
    history = []
    for step in range(1, max_steps + 1):
        decision = policy(question, history)
        if show:
            print(f"[{step}] 💭 {decision['thought']}")
        if "final" in decision:
            if show:
                print(f"    ✅ FINAL: {decision['final']}")
            return decision["final"], history
        name, args = decision["action"], decision["args"]
        if name not in tools:
            result = dict(ok=False, text=f"ERROR: there is no tool called {name!r}")
        elif name in needs_approval and not approve(name, args):
            result = dict(ok=False, text="DENIED: a human reviewer did not approve this action")
        else:
            result = tools[name](**args)
        history.append(dict(action=name, args=args, **result))
        if show:
            print(f"    🔧 {name}({', '.join(f'{k}={v!r}' for k, v in args.items())})\n    👁  {result['text'][:180]}")
    if show:
        print(f"    ⛔ stopped: step limit of {max_steps} reached without an answer")
    return None, history


CELL_SYNONYMS = {"monocytes": "Monocyte", "monocyte": "Monocyte", "b cells": "B", "nk cells": "NK",
                 "cd4 t cells": "CD4 T", "cd8 t cells": "CD8 T", "dendritic cells": "Dendritic"}


def literature_support(genes, docs, stim, cell_word):
    """For each gene: ids of retrieved abstracts with a sentence stating it is induced by `stim` in `cell_word`."""
    return {g: sorted({d["id"] for d in docs for s in d["sentences"] if re.search(rf"\b{g}\b", s)
                       and stim in s.lower() and cell_word in s.lower() and "induc" in s.lower()}) for g in genes}


def scripted_scientist(question, history):
    """A rule-based stand-in for an LLM policy: confirm the cell type, run DE, search the literature, cross-reference."""
    q = question.lower()
    cell_word = next((w for w in CELL_SYNONYMS if w in q), None)
    stim = next((s for s in ("interferon", "lps", "anti-cd3") if s in q), None)
    last = history[-1] if history else None
    done = [h["action"] for h in history if h["ok"]]
    if cell_word is None or stim is None:
        return dict(thought="The question does not name a cell type and a stimulus I know.",
                    final="Please specify the cell type and the stimulus.")
    cell = CELL_SYNONYMS[cell_word]
    if last is not None and not last["ok"]:
        return dict(thought=f"The last action failed ({last['text'][:60]}…). I report the failure instead of guessing.",
                    final=f"I could not complete the analysis: {last['text']}")
    if "lookup_marker" not in done:
        return dict(thought=f"Map '{cell_word}' to the annotation name and note its markers.", action="lookup_marker",
                    args=dict(cell_type=cell))
    if "run_differential_expression" not in done:
        return dict(thought="Find the responding genes in our own data, testing donors (not cells) as replicates.",
                    action="run_differential_expression", args=dict(cell_type=cell, stimulus=stim))
    de = next(h for h in history if h["action"] == "run_differential_expression" and h["ok"])["data"]
    if "search_literature" not in done:
        return dict(thought="Check which responding genes the literature supports.", action="search_literature",
                    args=dict(query=f"{stim} induced genes in {cell_word} " + " ".join(de.gene[:5])))
    docs = next(h for h in history if h["action"] == "search_literature")["data"]
    backed = {g: ids for g, ids in literature_support(de.gene, docs, stim, cell_word).items() if ids}
    novel = [g for g in de.gene if g not in backed]
    return dict(thought="Cross-reference: a gene counts as supported only if a retrieved sentence states it for this "
                        "stimulus and cell type.",
                final=f"{len(de)} genes respond to {stim} in {cell_word} (pseudobulk, q < 0.05). Literature-supported "
                      f"({len(backed)}): " + ", ".join(f"{g} [{ids[0]}]" for g, ids in backed.items())
                      + f". Without support in the retrieved literature ({len(novel)}): {', '.join(novel[:6])}, …")

In [ ]:
# ── 6.3 Run the agent on the question ──
question = "Which genes respond to interferon in monocytes, and is there literature support?"
final, history = run_agent(question, scripted_scientist)

In [ ]:
# ── 6.4 🧪 Ground truth: is the agent's answer right? ──
de_hits = next(h for h in history if h["action"] == "run_differential_expression")["data"]
true_resp = lab["true_logfc"].loc["Monocyte"]
true_set = set(true_resp.index[true_resp != 0])
found = set(de_hits.gene)
lit_truth = {g for d in library for (s, c, g) in d["facts"] if s == "interferon" and c == "Monocyte"}
docs_seen = next(h for h in history if h["action"] == "search_literature")["data"]
backed = {g for g, ids in literature_support(de_hits.gene, docs_seen, "interferon", "monocytes").items() if ids}
print(f"DE genes reported: {len(found)}; truly responding in monocytes: {len(found & true_set)} "
      f"(precision {len(found & true_set) / len(found):.2f}); recall {len(found & true_set) / len(true_set):.2f} "
      f"of the {len(true_set)} true responders")
print(f"interferon genes found: {sorted(found & set(INTERFERON_GENES))}")
print(f"literature-supported according to the agent: {sorted(backed)}")
print(f"genes with a stated (interferon, monocyte) fact anywhere in the library: {sorted(lit_truth)}")

🔍 **What just happened?** The log reads like a careful analyst. The agent confirmed the cell-type name, ran a **pseudobulk**
test with donors as replicates (the lesson of [Module 24](24_glm_count_data.ipynb) and [Module 42](42_trustworthy_ml.ipynb),
built into the tool), searched the literature with the top genes, and cross-referenced claims sentence by sentence. 🧪 Against
the simulator's truth, all 30 reported genes really respond in monocytes (precision 1.00) — all 10 interferon genes and 20
of the 40 monocyte-specific genes — i.e. 60 % of the 50 true responders; the others have effects too small for this
5-vs-5-donor design. Six genes are marked as literature-supported (IFIT1, IFIT3, IRF7, MX1, OAS1, STAT1); the rest — including
all the monocyte-specific genes, which the simulator names anonymously — have no support in the retrieved abstracts. That is an
honest answer, but "no support *found*" is not "no support *exists*": ISG15 *is* stated for monocytes in the library, just
not in the 8 abstracts retrieved, and 3 of the 10 interferon genes are never mentioned in our synthetic literature at all.

Note what made the answer trustworthy: not the policy's "intelligence" but the **tools** — a statistically sound test with
built-in design checks, and a claim-level literature check. An LLM agent is only as reliable as the tools and checks it is
given.

**Breaking the agent on purpose.** Four classic failure modes and the guardrail that catches each:

In [ ]:
# ── 6.5 Failure modes and guardrails ──
print("(a) WRONG ARGUMENT — a policy that passes the user's words straight to the tool:")
naive_policy = lambda q, h: (dict(thought="Run DE on 'monocytes'.", action="run_differential_expression",
                                  args=dict(cell_type="monocytes", stimulus="interferon")) if not h else
                             dict(thought="The tool said the name is invalid; it lists 'Monocyte'.", final="(retry with 'Monocyte')"))
_ = run_agent(question, naive_policy)

print("\n(b) LOOP — a policy that keeps searching for literature on a gene that no abstract mentions:")
stubborn = lambda q, h: dict(thought="Still no paper on Gene0042 — search again with other words.", action="search_literature",
                             args=dict(query=f"Gene0042 interferon monocytes attempt {len(h) + 1}"))
_ = run_agent(question, stubborn, max_steps=4)

print("\n(c) INVALID DESIGN — the same question on an experiment in which condition and batch are confounded:")
confounded = simulate_scrna(np.random.default_rng(SEED + 62), n_cells=2000, n_genes=300, n_donors=10, confounded=True)
print("   ", tool_differential_expression("Monocyte", "interferon", data=confounded)["text"])

print("\n(d) ACTIONS IN THE WORLD — ordering an experiment needs human approval:")
TOOLS_LAB = TOOLS | {"order_crispr_knockout": lambda gene: dict(ok=True, text=f"order placed for a {gene} knockout")}
eager = lambda q, h: (dict(thought="Order a knockout of the top gene to test causality.", action="order_crispr_knockout",
                           args=dict(gene="ISG15")) if not h else dict(thought="Report the decision.", final=h[-1]["text"]))
_ = run_agent(question, eager, tools=TOOLS_LAB, needs_approval=("order_crispr_knockout",), approve=lambda name, args: False)

🔍 **What just happened?** (a) The tool rejected "monocytes" with an **informative error** listing the valid names, which
lets a policy (or an LLM) recover instead of failing silently or, worse, quietly analysing the wrong cells. (b) The stubborn
policy would have searched forever; the **step limit** stopped it after 4 calls — real agents also need budgets for time,
money and API calls. (c) The DE tool **refused** an experiment whose conditions are confounded with batches
([Module 31](31_batch_integration.ipynb)): a validation guardrail encoded in the tool, so no policy can bypass it. (d) An
action with consequences in the physical world was **blocked** by a human-approval gate. Each guardrail lives *outside* the
language model, in code that can be tested.

**Plugging in a real LLM.** In a real agent the scripted policy is replaced by a call to a language model. The structure
stays the same — and so should the guardrails. As **pseudo-code only** (not executed here; names are placeholders, not a
specific provider's API):

```python
# PSEUDO-CODE — illustrative only; never paste confidential or patient data into a hosted model
SYSTEM = "You are a careful research assistant. Use the tools; cite evidence; say 'I don't know' when unsupported."
TOOL_SPECS = [describe(tool) for tool in TOOLS.values()]          # name, purpose, argument schema of each tool

def llm_policy(question, history):
    messages = [system(SYSTEM), user(question)] + [as_messages(h) for h in history]
    reply = llm_client.generate(model=MODEL_NAME, messages=messages, tools=TOOL_SPECS, temperature=0)
    if reply.tool_call:                                             # the model chose an action
        return dict(thought=reply.text, action=reply.tool_call.name, args=validate(reply.tool_call.arguments))
    return dict(thought=reply.text, final=reply.text)

final, log = run_agent(question, llm_policy, max_steps=8, needs_approval=ACTIONS_WITH_SIDE_EFFECTS, approve=ask_a_human)
save_for_provenance(log, model=MODEL_NAME, date=today())          # keep the full trace with the result
```

✏️ **Your turn.** Ask the scripted agent *"Which genes respond to LPS in monocytes, and is there literature support?"*. Which
tool fails, how does the policy react, and what would a careless policy have done instead?

### 📄 Research connection — Retrieval, tools and agents

- **What the field does.** Retrieval-augmented generation (Lewis *et al.* 2020) combined a neural retriever with a generator. ReAct (Yao
    *et al.* 2023) interleaved reasoning traces with tool calls; Toolformer (Schick *et al.* 2023) trained a model to decide
    when to call tools. PaperQA (Lála *et al.* 2023, preprint) applied retrieval and agentic search to scientific literature
    with citations. Ji *et al.* (2023) surveyed hallucination in language generation.
- **What this notebook builds.** TF-IDF and LSA retrieval with precision@k against known topics, a template RAG composer with claim-level
    verification, and a ReAct loop with three tools, a scripted policy, a step limit, informative tool errors, a design
    validator and a human-approval gate — all evaluated against simulated ground truth. Production agents use LLM policies,
    dense or hybrid retrieval and many more tools, but the same loop.
- **Where the frontier is.** Evaluating agents is hard: success on a benchmark task may not transfer, logs can look convincing while a
    step was wrong, and costs (time, money, laboratory actions) matter. Active topics (2024–2025) include benchmarks for
    biomedical research tasks, verification of generated claims against sources, multi-agent systems, and safety measures
    for agents that can act in the physical world.
- **Read next.** Lewis *et al.* 2020, *Retrieval-augmented generation for knowledge-intensive NLP tasks* (NeurIPS); Yao *et al.* 2023,
    *ReAct* (ICLR); Lála *et al.* 2023, *PaperQA* (arXiv preprint).

---
## 7 · 🧪 Biological foundation models — and how to evaluate them fairly

> 📍 **Where we are.** A **foundation model** is a large model pre-trained once, without labels, on broad data, and then
> reused for many downstream tasks — by probing its embeddings, fine-tuning it, or using its likelihoods "zero-shot".
> Biology has them for proteins (ESM-2: Lin *et al.* 2023), DNA (Nucleotide Transformer, Evo) and single cells
> (Geneformer: Theodoris *et al.* 2023; scGPT: Cui *et al.* 2024). Are they always better than simple baselines? Two careful
> evaluations say: not necessarily. Kedzierska *et al.* (2025) found that zero-shot embeddings of single-cell foundation
> models were often outperformed by simple baselines (highly variable genes, scVI) for cell-type clustering and batch
> integration; Ahlmann-Eltze, Huber & Anders (2025) found that deep models did not beat deliberately simple linear baselines
> at predicting the effects of genetic perturbations ([Module 41](41_perturbation_causal_inference.ipynb)).

We run a small, fair comparison where the truth is known. We sample a protein family from a Potts model with chemically
structured couplings (the sampler of [Module 26](26_attention_transformers.ipynb), copied below), pre-train a masked
language model on 1,400 family members, and ask it to predict the **fitness** (the true Potts score) of 600 *other* members
from few labels. Rules of a fair comparison: (1) the downstream sequences were never seen in pre-training; (2) every
representation gets the same downstream model (ridge regression with the penalty chosen by internal cross-validation) and
the same training and test sequences; (3) strong simple baselines (one-hot encoding, PCA); (4) several random label sets,
reported with their spread.

In [ ]:
# ── 7.1 A SIMULATED protein family (sampler copied from Module 26) ──
AMINO = "ACDEFGHIKLMNPQRSTVWY"                                              # the order used by the toolkit
HYDROPATHY = np.array([1.8, 2.5, -3.5, -3.5, 2.8, -0.4, -3.2, 4.5, -3.9, 3.8, 1.9, -3.5, -1.6, -3.5, -4.5, -0.8, -0.7, 4.2, -0.9, -1.3])
CHARGE = np.array([0, 0, -1, -1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0], float)
VOLUME = np.array([88.6, 108.5, 111.1, 138.4, 189.9, 60.1, 153.2, 166.7, 168.6, 166.7, 162.9, 114.1, 112.7, 143.8, 173.4,
                   89.0, 116.1, 140.0, 227.8, 193.6])


def zscore(v):
    """Centre a vector and divide by its standard deviation."""
    return (v - v.mean()) / v.std()


def chemistry_matrices():
    """Three 20 × 20 residue–residue preference matrices (hydrophobic packing, salt bridges, size complementarity)."""
    raw = {"hydrophobic packing": np.outer(zscore(HYDROPATHY), zscore(HYDROPATHY)),
           "salt bridge": -np.outer(CHARGE, CHARGE), "size complementarity": -np.outer(zscore(VOLUME), zscore(VOLUME))}
    out = {}
    for name, M in raw.items():
        M = M - M.mean(0, keepdims=True) - M.mean(1, keepdims=True) + M.mean()
        out[name] = M / np.abs(M).max()
    return out


CHEM = chemistry_matrices()


def sample_chemistry_family(rng, contacts, n_seqs=3000, n_sweeps=60, strength=2.0, noise=0.1):
    """Gibbs-sample an MSA from a Potts model whose couplings mix the three chemistry matrices (Module 26, cell 1.3)."""
    L, q = contacts.shape[0], 20
    mats = np.stack(list(CHEM.values()))
    h = rng.normal(0, 1, (L, q)) * rng.uniform(0.3, 1.5, L)[:, None]
    J, w = np.zeros((L, L, q, q)), np.zeros((L, L, 3))
    for i, j in zip(*np.nonzero(np.triu(contacts, 1))):
        w[i, j] = w[j, i] = rng.dirichlet([0.5, 0.5, 0.5]) * rng.uniform(0.5, 1.5)
        J[i, j] = strength * np.tensordot(w[i, j], mats, axes=1) + rng.normal(0, noise, (q, q))
        J[j, i] = J[i, j].T
    neighbours = [np.flatnonzero(contacts[i]) for i in range(L)]
    x, rows, trace = rng.integers(0, q, (n_seqs, L)), np.arange(n_seqs), []
    for _ in range(n_sweeps):
        for i in rng.permutation(L):
            logits = np.tile(h[i], (n_seqs, 1))
            for j in neighbours[i]:
                logits += J[j, i][x[:, j]]
            x[rows, i] = np.argmax(logits - np.log(-np.log(rng.random((n_seqs, q)))), axis=1)   # Gumbel-max sampling
        trace.append(potts_score(x, h, J).mean())
    wild_type = x[np.argmax(potts_score(x, h, J))].copy()
    return dict(msa=x, h=h, J=J, weights=w, wild_type=wild_type, trace=np.array(trace))


t0 = time.time()
contacts = simulate_helix_bundle()["contacts"]
L_PROT = contacts.shape[0]
fam = sample_chemistry_family(np.random.default_rng(SEED + 71), contacts, n_seqs=2000, n_sweeps=40)
fitness = potts_score(fam["msa"], fam["h"], fam["J"])                       # TRUE fitness of every member
order = np.random.default_rng(SEED + 72).permutation(len(fitness))
pre_idx, down_idx = order[:1400], order[1400:]                              # pre-training vs downstream (never overlap)
print(f"SIMULATED family: {len(fitness)} sequences × {L_PROT} residues, sampled in {time.time() - t0:.1f} s; "
      f"{len(pre_idx)} for pre-training, {len(down_idx)} for the downstream task")

In [ ]:
# ── 7.2 Pre-train a small masked protein language model (≈ 10 s) ──
MASK_ID = 20


class ProteinMLM(nn.Module):
    """Embeddings → 2 transformer encoder layers (PyTorch's built-in version of Module 26's block) → residue logits."""

    def __init__(self, d_model=32, n_heads=4, n_layers=2):
        super().__init__()
        self.tok, self.pos = nn.Embedding(21, d_model), nn.Embedding(L_PROT, d_model)
        self.layers = nn.ModuleList([nn.TransformerEncoderLayer(d_model, n_heads, 2 * d_model, dropout=0.0,
                                                                batch_first=True, norm_first=True) for _ in range(n_layers)])
        self.ln, self.head = nn.LayerNorm(d_model), nn.Linear(d_model, 20)

    def forward(self, x):
        z = self.tok(x) + self.pos.weight
        for layer in self.layers:
            z = layer(z)
        z = self.ln(z)
        return self.head(z), z                                            # logits and per-residue embeddings


torch.manual_seed(SEED)
mlm, t0 = ProteinMLM(), time.time()
opt = torch.optim.AdamW(mlm.parameters(), lr=3e-3, weight_decay=0.01)
X_pre = torch.tensor(fam["msa"][pre_idx])
for step in range(300):
    batch = X_pre[torch.randint(0, len(X_pre), (64,))]
    hidden = torch.rand(batch.shape) < 0.15                                # mask 15 % of residues
    x_in = batch.clone(); x_in[hidden] = MASK_ID
    loss = F.cross_entropy(mlm(x_in)[0][hidden], batch[hidden])
    opt.zero_grad(); loss.backward(); opt.step()
mlm.eval()
print(f"masked-LM pre-training: 300 steps in {time.time() - t0:.1f} s, final masked-residue loss {loss.item():.2f} nats "
      f"(uniform guess: {np.log(20):.2f})")
X_down = torch.tensor(fam["msa"][down_idx])
TEST7 = np.arange(400, 600)                                                # fixed test sequences (down_idx is shuffled)
with torch.no_grad():
    emb_mean = mlm(X_down)[1].mean(1).numpy()                              # one 32-number embedding per sequence
    X_te = X_down[TEST7]
    pll = np.zeros(len(X_te))                                              # zero-shot: pseudo-log-likelihood
    for i in range(L_PROT):                                                # mask one position at a time
        x_in = X_te.clone(); x_in[:, i] = MASK_ID
        pll += torch.log_softmax(mlm(x_in)[0][:, i], -1).gather(1, X_te[:, i:i + 1])[:, 0].numpy()

In [ ]:
# ── 7.3 🧪 A fair comparison: same labels, same ridge model, same test sequences ──
y_fit = fitness[down_idx]
onehot = np.eye(20)[fam["msa"][down_idx]].reshape(len(down_idx), -1)        # 68 × 20 = 1,360 features
pcs, _, _ = pca(onehot, 32)
REPS = {"one-hot (1,360 features)": onehot, "PCA of one-hot (32)": pcs, "masked-LM embedding (32)": emb_mean}
rng = np.random.default_rng(SEED + 73)
results = []
for n_lab in (30, 100, 300):
    for rep in range(5):
        tr = rng.choice(400, n_lab, replace=False)                          # labels drawn from the first 400 sequences
        for name, Xf in REPS.items():
            model = RidgeCV(alphas=np.logspace(-2, 4, 13)).fit(Xf[tr], y_fit[tr])
            results.append((name, n_lab, stats.spearmanr(model.predict(Xf[TEST7]), y_fit[TEST7])[0]))
res = pd.DataFrame(results, columns=["representation", "labelled sequences", "Spearman"])
summary7 = res.groupby(["representation", "labelled sequences"])["Spearman"].agg(["mean", "std"]).unstack()
print(summary7.round(3).to_string())
zero_shot = stats.spearmanr(pll, y_fit[TEST7])[0]
print(f"\nzero-shot pseudo-log-likelihood (no labels at all): Spearman {zero_shot:.3f}")

fig, ax = plt.subplots(figsize=(7, 3.6))
for (name, c) in zip(REPS, [OKABE_ITO[3], OKABE_ITO[4], OKABE_ITO[0]]):
    m, s = summary7.loc[name, "mean"], summary7.loc[name, "std"]
    ax.errorbar(np.arange(3), m.values, yerr=s.values, marker="o", capsize=3, color=c, label=name)
ax.axhline(zero_shot, color=OKABE_ITO[1], ls="--", label="masked-LM zero-shot likelihood (0 labels)")
ax.set_xticks(range(3)); ax.set_xticklabels(["30", "100", "300"]); ax.set_ylim(0, 1)
ax.set_xlabel("labelled sequences for the ridge model"); ax.set_ylabel("Spearman ρ with TRUE fitness")
ax.set_title("SIMULATED protein family: how a foundation model is used matters"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The result is more interesting than "the foundation model wins".

* **Frozen embeddings + a linear probe do not beat simple baselines.** With 30 labelled sequences the 32-number masked-LM
  embedding is slightly ahead (Spearman 0.68 vs 0.63 for one-hot and for PCA — less than one standard deviation of the spread
  over label sets); with 100 labels all three are tied (0.72–0.73); with 300 labels plain one-hot ridge is clearly best (0.81
  vs 0.75 for the embedding). Averaging the residue embeddings into one vector throws away *which* residue sits *where*, and
  fitness depends on exactly that; one-hot keeps it and can use it once there are enough labels.
* **The zero-shot likelihood is the best of all.** Without a single label, the model's pseudo-log-likelihood ranks the test
  sequences by fitness with Spearman 0.93 — better than every supervised model with 300 labels. The pre-training objective
  (predict residues of family members) is closely aligned with this task, because in our simulation fitness *is* the energy
  of the distribution the family was sampled from.

Both findings echo real evaluations: how a foundation model is used (zero-shot likelihood, frozen embeddings, fine-tuning)
matters as much as the model, and a claim of superiority needs strong baselines, identical splits and error bars. Our
zero-shot success is also a warning: it works because the label is aligned with the pre-training distribution. For a
property that evolution did not optimise — activity on a new substrate, expression in a foreign host — likelihoods can be
weakly correlated or even misleading.

⚠️ **Leakage, foundation-model edition.** A public model may have been pre-trained on your test sequences (or their close
homologues), and a single-cell model on the very atlas you evaluate on. Ask what the pre-training data contained before
interpreting a benchmark ([Module 42](42_trustworthy_ml.ipynb), Section 2).

✏️ **Your turn.** Replace the mean-pooled embedding by the concatenation of all per-residue embeddings
(`mlm(X_down)[1].reshape(len(X_down), -1)`, 68 × 32 features). Does the probe now catch up with one-hot at 300 labels? Why
might it?

### 📄 Research connection — Biological foundation models and their evaluation

- **What the field does.** Protein language models (ESM: Rives *et al.* 2021; ESM-2/ESMFold: Lin *et al.* 2023), DNA models (Nucleotide
    Transformer: Dalla-Torre *et al.* 2025; Evo: Nguyen *et al.* 2024) and single-cell models (Geneformer: Theodoris *et al.*
    2023; scGPT: Cui *et al.* 2024) are pre-trained on millions to billions of sequences or cells and reused by zero-shot
    scoring, probing or fine-tuning. LLMs have also been evaluated on medical knowledge (Med-PaLM: Singhal *et al.* 2023).
- **What this notebook builds.** A masked protein language model pre-trained on a simulated family, compared under identical splits with one-hot
    and PCA baselines for few-label fitness prediction, plus a zero-shot pseudo-log-likelihood — with the true fitness known.
- **Where the frontier is.** Evaluation is the open problem: Kedzierska *et al.* (2025) found zero-shot single-cell embeddings often beaten by
    simple baselines, and Ahlmann-Eltze, Huber & Anders (2025) found deep perturbation-response models not better than linear
    baselines. Benchmarks with documented splits (ProteinGym for variant effects), honest reporting of what was in the
    pre-training data, and tasks that test genuine generalisation are active areas, as is understanding when scale helps.
- **Read next.** Kedzierska *et al.* 2025, *Zero-shot evaluation reveals limitations of single-cell foundation models* (Genome Biol);
    Ahlmann-Eltze, Huber & Anders 2025 (Nat Methods), deep-learning vs linear baselines for perturbation prediction;
    Lin *et al.* 2023, *ESM-2/ESMFold* (Science).

---
## 8 · AI scientists and lab automation: discovery at scale — including false discovery

> 📍 **Where we are.** Agents that search, analyse and plan can be chained into systems that run parts of research
> autonomously. Coscientist (Boiko *et al.* 2023) designed and executed chemical reactions on automated laboratory
> hardware; ChemCrow (Bran *et al.* 2024) combined an LLM with expert chemistry tools; the Virtual Lab (Swanson *et al.*
> 2025) organised LLM agents into a "team" that designed nanobodies, with experimental validation by human scientists; the AI
> co-scientist (Gottweis *et al.* 2025, preprint) generates, debates and ranks hypotheses. What does automation do to the
> error rate of science?

**An automated analyst in a world with no effects.** We simulate a single-cell study in which the "stimulation" does
**nothing** (`stim_effect=0`): every true effect is zero. An automated analyst tests every gene in every cell type. Version 1
does what a fast but careless pipeline (or a poorly supervised agent) might do: compare *cells* between conditions with a
Wilcoxon test and report everything with $p<0.05$. Version 2 has statistical guardrails: donors as replicates (pseudobulk)
and Benjamini–Hochberg control of the false discovery rate.

In [ ]:
# ── 8.1 🧪 Automation scales false discoveries: a null world (no true effects at all) ──
null_lab = simulate_scrna(np.random.default_rng(SEED + 81), n_cells=4000, n_genes=300, n_donors=10, stim_effect=0.0)
obs0, C0 = null_lab["obs"], null_lab["counts"]
logn = np.log1p(1e4 * C0 / C0.sum(1, keepdims=True))
cond0 = obs0["condition"].to_numpy(str) == "stimulated"
naive_hits, careful_hits = 0, 0
for ct in TYPE_NAMES:
    m = obs0["cell_type"].to_numpy(str) == ct
    expressed = (C0[m] > 0).mean(0) > 0.05
    p_cells = stats.mannwhitneyu(logn[m & cond0][:, expressed], logn[m & ~cond0][:, expressed], axis=0).pvalue
    naive_hits += int((p_cells < 0.05).sum())                                # cells as replicates, no correction
    careful = tool_differential_expression(ct, "interferon", data=null_lab)  # pseudobulk + BH (Section 6 tool)
    careful_hits += len(careful["data"]) if careful["ok"] else 0
n_tests = len(TYPE_NAMES) * 300
print(f"TRUE effects in this simulation: 0")
print(f"naive automated analyst (cell-level tests, p < 0.05, no correction): {naive_hits} 'discoveries'")
print(f"guardrailed analyst (donor pseudobulk, BH q < 0.05, |log2FC| ≥ 1): {careful_hits} 'discoveries'")
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar([0, 1], [naive_hits, careful_hits], color=[OKABE_ITO[1], OKABE_ITO[2]])
for x, v in enumerate([naive_hits, careful_hits]):
    ax.text(x, v + 5, str(v), ha="center")
ax.set_xticks([0, 1]); ax.set_xticklabels(["naive automation", "with statistical guardrails"])
ax.set_ylim(0, 1.2 * max(naive_hits, 1)); ax.set_ylabel("claimed discoveries")
ax.set_title("SIMULATED null world: every claimed discovery is false")
plt.tight_layout(); plt.show()

🔍 **What just happened?** In a world with no effects, the naive analyst reports 405 "discoveries" among at most 1,800 tests
— far more than the 5 % expected by chance alone, because cells from the same donor are not independent (donor differences masquerade as
treatment effects; [Module 42](42_trustworthy_ml.ipynb), Section 2a). The guardrailed analyst reports 0. An automated system
that runs thousands of analyses per day multiplies whatever its statistical habits are. Speed is not the bottleneck of good
science; **error control, replication and human scrutiny** are — and they must be designed into AI-scientist systems, not
hoped for.

**Evaluating AI scientists.** Published systems are usually evaluated on a handful of showcase tasks chosen by their
developers. Questions to ask of any claim: How many tasks were attempted and how were they chosen? What did humans do
(choose the goal, filter outputs, design the validation)? Were results validated experimentally and independently? Is the
code, prompt and model version available so that others can reproduce the run? Is there a comparison with a non-AI baseline
(a human expert, a simple search)?

**Biosecurity and dual use.** Tools that help design molecules or organisms can, in principle, be misused. Urbina *et al.*
(2022) showed that a generative model built to *avoid* toxicity could be redirected to *propose* toxic molecules — a
warning that the authors published precisely to prompt safeguards. Responsible practice includes: not publishing or
sharing capabilities whose main use is harm; screening synthesis orders (as DNA-synthesis providers do); keeping humans in
the loop for physical actions; following institutional biosafety rules; and asking, before building a tool, who else could
use it and for what. (This notebook deliberately keeps all such discussion conceptual.)

**Privacy.** Text pasted into a hosted model leaves your institution. Patient data, identifiable genomes, unpublished
results and confidential reviews must not be entered into tools that are not approved for them; de-identification of
genomic data is hard, because genomes are inherently identifying.

✏️ **Your turn.** Give the naive analyst a Benjamini–Hochberg correction but keep the cell-level test. How many false
discoveries remain? What does this tell you about which guardrail matters most here?

### 📄 Research connection — AI for scientific discovery and its evaluation

- **What the field does.** LLM-based systems now plan experiments (Coscientist: Boiko *et al.* 2023), use expert tools (ChemCrow: Bran *et
    al.* 2024), collaborate as role-playing agents (Virtual Lab: Swanson *et al.* 2025), generate hypotheses (AI co-scientist:
    Gottweis *et al.* 2025, preprint), and answer medical questions (Med-PaLM: Singhal *et al.* 2023). Dual-use risks were
    highlighted by Urbina *et al.* (2022).
- **What this notebook builds.** A ReAct agent with guarded tools (Section 6) and an automated analyst in a null world, showing that statistical
    guardrails — replicates at the right level, FDR control — decide whether automation produces discoveries or noise.
- **Where the frontier is.** How to evaluate autonomous research systems fairly (task selection, human contribution, reproducibility,
    experimental validation), how to keep humans meaningfully in control of physical experiments, and how to govern dual-use
    capabilities are open questions that the community is actively debating (as of 2025). Claims of autonomous discovery
    should be read with the scepticism of [Module 42](42_trustworthy_ml.ipynb).
- **Read next.** Boiko *et al.* 2023, *Autonomous chemical research with large language models* (Nature); Swanson *et al.* 2025,
    *The Virtual Lab* (Nature); Urbina *et al.* 2022, *Dual use of artificial-intelligence-powered drug discovery* (Nat Mach Intell).

---
## 9 · 🧭 From toy to real systems: using LLMs responsibly in research

> 📍 **Where we are.** Every component above has a real counterpart that is millions of times larger. Here is what
> changes, and a checklist for everyday use — no downloads, just orientation.

**Real models and resources.** General-purpose LLMs are accessed through hosted APIs or run locally as open-weight models
(the [Hugging Face](https://huggingface.co) hub hosts many, with the `transformers` library). Literature: [PubMed](https://pubmed.ncbi.nlm.nih.gov)
and [Europe PMC](https://europepmc.org) provide search APIs; PaperQA-style tools add retrieval and citation. Biological
foundation models: ESM ([github.com/facebookresearch/esm](https://github.com/facebookresearch/esm)), Evo, Nucleotide
Transformer, Geneformer and scGPT, often evaluated on [ProteinGym](https://proteingym.org) or atlases such as
[CELLxGENE](https://cellxgene.cziscience.com). Training such models takes large GPU clusters; *using* them (embedding,
scoring, retrieval, calling an API) is feasible on a laptop or a free Colab GPU.

**What changes with scale.** Real LLMs are trained on human text, so their "fact table" is the internet's: often right,
sometimes outdated, sometimes wrong, and with no record of where a statement came from. Their fluency is far greater than
TinyGPT's, which makes errors harder to spot. Their training data are unknown, so benchmark results may be inflated by
contamination.

**A checklist for using LLMs in your research.**

1. **Verify every citation, number and claim** against the primary source. Fabricated references and plausible but wrong
   numbers are common failure modes.
2. **Keep provenance**: record the model, version, date, prompt and output of anything that enters your work, as you would
   for a reagent lot.
3. **Review generated code like a stranger's code**: read it, test it on data with a known answer (simulations, as in this
   course), and check edge cases. Silent errors are worse than crashes.
4. **Never paste sensitive data** (patient records, identifiable genomes, unpublished or confidential material) into a tool
   that is not approved for it.
5. **Prefer grounded workflows**: retrieval with visible sources, tools that do the computation, and abstention when evidence
   is missing.
6. **Report LLM use** according to your journal's and institution's policies; you remain responsible for the content.
7. **Beware of automation bias**: a confident answer is not a correct one (Section 2), and an agent's tidy log is not proof
   that each step was right (Section 6).

**The first three things that go wrong.** (1) Hallucinated references or facts that slip into a manuscript or grant.
(2) Plausible code that quietly does the wrong analysis (wrong normalisation, cells as replicates, leakage). (3) Over-trust in
a foundation model or agent benchmark whose test data overlapped its training data or whose baselines were weak.

## 📌 Key takeaways

- A **language model** is a next-token predictor, $P(w_1..w_T)=\prod_t P(w_t\mid w_{<t})$, trained by cross-entropy;
  perplexity $=e^{H}$. n-grams are Markov chains whose context is truncated; transformers can use the whole context.
- On our SIMULATED notebook, TinyGPT matched the true grammar almost exactly (perplexity 1.66 vs 1.65), learning the hidden
  fact table that no n-gram could; loss fell smoothly with model size.
- **Hallucination** is fluent but ungrounded generation: asked about experiments never written down, the model gave
  guesses (one right, one wrong) in the same confident form as its facts.
- **Tokenisation** shapes what a model sees: BPE on DNA turned planted motifs into tokens. Assistants add instruction tuning
  and preference optimisation (a logistic model of human comparisons) on top of pre-training.
- **Retrieval** (TF-IDF, embeddings) grounds answers in documents and can be evaluated with precision@k; **RAG** fails when
  retrieval fails — a synonym produced a confident, cited, wrong answer — so claims must be verified against the text.
- **Agents** (ReAct) loop over thought → action → observation. Their reliability comes from tools with built-in checks,
  informative errors, step limits, design validation and human approval for real-world actions.
- **Foundation models** must be compared with strong baselines under identical splits: frozen embeddings gave at most a small
  edge with 30 labels and lost to one-hot ridge with 300, while a zero-shot likelihood aligned with the task beat both.
- **AI scientists** scale false discoveries as easily as true ones; statistical guardrails, replication, provenance,
  privacy and biosecurity considerations are part of responsible use.

---
## ✅ Checkpoint

1. What is the perplexity of a model that assigns uniform probability to a vocabulary of 54 words? Of a model that always gives the correct next word probability 0.5?
   <details><summary>Answer</summary>Uniform: $e^{\log 54}=54$. Probability 0.5 for every correct word: cross-entropy $\log 2$, perplexity 2.</details>
2. What does the causal mask do, and what would go wrong if you trained a GPT without it?
   <details><summary>Answer</summary>It sets attention scores to future positions to $-\infty$, so position $t$ only sees tokens $\le t$. Without it the model could read the token it is supposed to predict; training loss would drop to near zero, but the model could not generate text, where future tokens do not exist yet.</details>
3. Why could no n-gram model predict the 'result' word of our notebook entries, while TinyGPT could?
   <details><summary>Answer</summary>The result depends on the cell type, stimulus and gene, which appear up to 14 tokens earlier. An n-gram only sees the last n − 1 tokens (and long contexts are too sparse to count); attention can look back at any earlier token.</details>
4. Why does retrieval-augmented generation reduce but not remove hallucination? Give two failure points.
   <details><summary>Answer</summary>The generator is grounded in retrieved text, so it invents less — but errors can enter through the query (vocabulary mismatch, as with 'IFN-beta'), the retriever (wrong or missing documents), the documents (outdated, wrong) and the generator (misreading, citing a paper for a claim it does not make).</details>
5. Name three guardrails for a tool-using agent and the failure each prevents.
   <details><summary>Answer</summary>Informative input validation in tools (wrong arguments), step/budget limits (loops), design checks inside analysis tools (invalid statistics, e.g. confounding), human approval for actions with real-world effects, claim-level verification (ungrounded answers).</details>
6. Why compare a foundation model with simple baselines, and what makes the comparison fair?
   <details><summary>Answer</summary>Large models are often not better than one-hot, PCA, HVG or linear baselines for a given task; fairness requires that the downstream data were not in pre-training, identical splits and downstream models for all representations, tuned baselines, and results over several random splits with their spread.</details>
7. Code reading: in `compose_answer`, what has to be true of a sentence for the verified composer to keep a claim?
   <details><summary>Answer</summary>The sentence must contain the gene (as a whole word), every stimulus and cell type found in the question, and an induction verb ('induc…'); if the question's stimulus or cell type is not recognised, nothing is supported and the composer abstains.</details>
8. In the null-world experiment, why did the naive analyst find many 'discoveries' although no effects exist?
   <details><summary>Answer</summary>It treated cells as independent replicates: cells of one donor share donor-specific expression differences, and since donors are nested in conditions those differences look like treatment effects; with thousands of tests and no FDR control, many pass p < 0.05. Pseudobulk by donor plus BH removes them.</details>

## 🧠 Questions & discussion

1. TinyGPT 'knew' the fact table because the corpus was consistent. Real text contains contradictory claims, retracted papers and outdated knowledge. How would you expect an LLM to behave on a question where the literature disagrees, and how would you design a system that surfaces the disagreement instead of hiding it?
2. Our agent's answer was trustworthy because its tools were. Who should be responsible for the correctness of an analysis run by an AI agent: the tool authors, the model provider, or the scientist who signs the paper?
3. A colleague proposes to let an AI system choose and order the next round of CRISPR experiments automatically to save time. Which decisions would you keep for humans, and which safety and biosafety checks would you require?
4. Zero-shot likelihoods worked well for fitness in our simulation. For which biological properties would you expect them to fail, and how would you find out before relying on them?
5. Hosted LLMs are convenient but send your text to a third party. Discuss the privacy, reproducibility (model versions change) and equity (cost, access) implications of building a research pipeline on them versus on open-weight models.

## 🚀 Bonus — going further

- **BM25 vs TF-IDF.** Implement BM25 (✏️ in Section 4) and evaluate both on harder queries that you write yourself, including paraphrases. Then add query expansion with a synonym table and measure the gain.
- **Fine-tune the tiny GPT on a sub-domain.** Continue training the d = 64 model for 200 steps on entries about interferon only. Does held-out loss on interferon entries fall while loss on LPS entries rises ('catastrophic forgetting')?
- **Evaluate an agent on a task suite.** Write ten scripted questions (different cell types, stimuli, malformed names, unsupported stimuli) with known correct outcomes, run the agent on all of them and report a success rate and the failure types — a miniature agent benchmark.
- **A real protein language model.** On Colab with the Hugging Face `transformers` library, compute zero-shot masked-marginal scores of the smallest ESM-2 model for a ProteinGym assay and compare them with a one-hot ridge baseline trained on 100 labelled variants, using the same split.
- **A real literature assistant.** Try an open-source RAG tool (e.g. PaperQA) on a question in your field and audit every citation in its answer. How many claims are fully supported by the cited passage?

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module: 43 · Foundation models, language models and AI agents for biology
Biological question: what is inside LLM-based tools, how do they fail, and how should a biologist evaluate them?
Data (simulated): lab-notebook corpus from a probabilistic grammar + hidden fact table (true token probabilities known);
                  200 synthetic abstracts (topics and facts known); simulate_scrna lab data (true responders known);
                  a Potts protein family (true fitness known); a null single-cell world (no true effects)
Method: n-grams; TinyGPT (causal transformer); BPE; TF-IDF/LSA retrieval; template RAG with verification;
        ReAct agent with guarded tools; masked protein LM vs baselines; automated analyst with/without guardrails
Key equation: P(w_1..w_T) = Π_t P(w_t | w_<t) ;  PPL = exp(H) ;  P(y_w ≻ y_l) = σ(r(y_w) − r(y_l))
Key parameters: context length, model size, temperature, number of BPE merges, k retrieved documents, step limit
Ground-truth check result: CE ladder (n-grams / TinyGPT / truth) = … ; P(up) on held-out combos = … ;
                           precision@5 = … ; agent precision/recall = … ; Spearman (one-hot / embedding / zero-shot) = …
What changed when I modified … (temperature / HELD_OUT / query wording / max_steps / labels): …
One failure mode: (e.g. confident wrong RAG answer after a retrieval miss; frozen embeddings losing to one-hot)
Connection to research: Vaswani 2017; Lewis 2020 (RAG); Yao 2023 (ReAct); Kedzierska 2025; tool: PaperQA, ESM-2
```

## 📚 References

- Shannon C. E. (1951) Prediction and entropy of printed English. *Bell Syst Tech J* 30:50–64.
- Bengio Y., Ducharme R., Vincent P., Jauvin C. (2003) A neural probabilistic language model. *J Mach Learn Res* 3:1137–1155.
- Mikolov T., Chen K., Corrado G., Dean J. (2013) Efficient estimation of word representations in vector space. arXiv:1301.3781 (preprint).
- Vaswani A. *et al.* (2017) Attention is all you need. *NeurIPS*.
- Brown T. B. *et al.* (2020) Language models are few-shot learners. *NeurIPS*.
- Kaplan J. *et al.* (2020) Scaling laws for neural language models. arXiv:2001.08361 (preprint).
- Hoffmann J. *et al.* (2022) Training compute-optimal large language models. *NeurIPS*.
- Ouyang L. *et al.* (2022) Training language models to follow instructions with human feedback. *NeurIPS*.
- Wei J. *et al.* (2022) Chain-of-thought prompting elicits reasoning in large language models. *NeurIPS*.
- Rafailov R. *et al.* (2023) Direct preference optimization: your language model is secretly a reward model. *NeurIPS*.
- Sennrich R., Haddow B., Birch A. (2016) Neural machine translation of rare words with subword units. *ACL*.
- Sparck Jones K. (1972) A statistical interpretation of term specificity and its application in retrieval. *J Doc* 28:11–21.
- Deerwester S., Dumais S. T., Furnas G. W., Landauer T. K., Harshman R. (1990) Indexing by latent semantic analysis. *J Am Soc Inf Sci* 41:391–407.
- Robertson S., Zaragoza H. (2009) The probabilistic relevance framework: BM25 and beyond. *Found Trends Inf Retr* 3:333–389.
- Lewis P. *et al.* (2020) Retrieval-augmented generation for knowledge-intensive NLP tasks. *NeurIPS*.
- Yao S. *et al.* (2023) ReAct: synergizing reasoning and acting in language models. *ICLR*.
- Schick T. *et al.* (2023) Toolformer: language models can teach themselves to use tools. *NeurIPS*.
- Lála J. *et al.* (2023) PaperQA: retrieval-augmented generative agent for scientific research. arXiv:2312.07559 (preprint).
- Ji Z. *et al.* (2023) Survey of hallucination in natural language generation. *ACM Comput Surv* 55:248.
- Singhal K. *et al.* (2023) Large language models encode clinical knowledge. *Nature* 620:172–180.
- Boiko D. A., MacKnight R., Kline B., Gomes G. (2023) Autonomous chemical research with large language models. *Nature* 624:570–578.
- Bran A. M. *et al.* (2024) Augmenting large language models with chemistry tools. *Nat Mach Intell* 6:525–535.
- Swanson K., Wu W., Bulaong N. L., Pak J. E., Zou J. (2025) The Virtual Lab of AI agents designs new SARS-CoV-2 nanobodies. *Nature*.
- Gottweis J. *et al.* (2025) Towards an AI co-scientist. arXiv:2502.18864 (preprint).
- Rives A. *et al.* (2021) Biological structure and function emerge from scaling unsupervised learning to 250 million protein sequences. *PNAS* 118:e2016239118.
- Lin Z. *et al.* (2023) Evolutionary-scale prediction of atomic-level protein structure with a language model. *Science* 379:1123–1130.
- Dalla-Torre H. *et al.* (2025) Nucleotide Transformer: building and evaluating robust foundation models for human genomics. *Nat Methods*.
- Nguyen E. *et al.* (2024) Sequence modeling and design from molecular to genome scale with Evo. *Science*.
- Theodoris C. V. *et al.* (2023) Transfer learning enables predictions in network biology. *Nature* 618:616–624.
- Cui H. *et al.* (2024) scGPT: toward building a foundation model for single-cell multi-omics using generative AI. *Nat Methods* 21:1470–1480.
- Kedzierska K. Z., Crawford L., Amini A. P., Lu A. X. (2025) Zero-shot evaluation reveals limitations of single-cell foundation models. *Genome Biol*.
- Ahlmann-Eltze C., Huber W., Anders S. (2025) Deep-learning-based gene perturbation effect prediction does not yet outperform simple linear baselines. *Nat Methods*.
- Urbina F., Lentzos F., Invernizzi C., Ekins S. (2022) Dual use of artificial-intelligence-powered drug discovery. *Nat Mach Intell* 4:189–191.

## ➡️ Next

You now have the tools of the whole course — from likelihoods and graphical models to transformers, conformal
prediction and agents. **[Module 44 · Capstone: from a question to a reproducible study](44_capstone_project.ipynb)**
puts them together: a question, an evaluation contract, honest comparisons and a reproducible report.